# Which Restaurants Serve the Best Version of Each Dish?

**Task 4 — Dish-Specific Restaurant Ranking with Evidence-Grounded Explanations**

Task 4 turns the curated dish vocabulary from Task 3 into a recommendation problem. Instead of asking which restaurant is generally good, the task asks which restaurants are recommended for a specific dish and what review evidence supports that recommendation.


## Table of Contents

1. [Runtime and Package Setup](#0.-Runtime-and-package-setup)
2. [Configuration](#1.-Configuration)
3. [Dataset Acquisition and Discovery](#2.-Dataset-acquisition-and-discovery)
4. [Load the Task 3 Dish List](#3.-Load-the-Task-3-dish-list)
5. [Build the Cuisine-Specific Review Corpus](#4.-Build-the-cuisine-specific-review-corpus)
6. [Dish Mention Extraction with Local Context](#5.-Dish-mention-extraction-with-local-context)
7. [Method A — Ratings-Only Baseline](#6.-Method-A-%E2%80%94-Ratings-only-baseline)
8. [Method B — Lexicon-Based Dish Sentiment](#7.-Method-B-%E2%80%94-Lexicon-based-dish-local-sentiment)
9. [Method C — Transformer Sentiment](#8.-Method-C-%E2%80%94-Transformer-sentiment-(RoBERTa),-with-honest-fallback)
10. [Compare Ranking Methods](#9.-Compare-ranking-methods)
11. [Qualitative Error Analysis](#10.-Qualitative-error-analysis)
12. [Required Comparison and Interpretation](#11.-Required-comparison-and-interpretation)
14. [Required Report Deliverables](#12.-Required-report-deliverables)


# Task 4 — Dish-Specific Restaurant Ranking with Evidence-Grounded Explanations

Task 4 ranks restaurants for selected dishes using local review evidence. It compares ratings-only, lexicon sentiment, and transformer/fallback sentiment, then turns top evidence snippets into short grounded explanations.

## Pipeline Artifact Handoff

Task 4 produces dish-local sentiment rankings and grounded recommendation explanations.
Its key handoff artifacts live under `artifacts/task4/`:
`dish_mention_evidence.csv`, `dish_restaurant_rankings_long.csv`,
`ranking_method_comparison.csv`, and `grounded_recommendation_explanations.csv`.

## Required artifact contract for Task 4

Your completed notebook should produce these core artifacts under `artifacts/task4/`:

- `loaded_task3_dish_list.csv` — the Task 3 dish list as loaded by Task 4.
- `dish_alias_table.csv` — aliases or canonical dish forms used for matching.
- `dish_mention_evidence.csv` — dish-local evidence rows with sentiment scores.
- `dish_mention_coverage.csv` — dish/business coverage diagnostics.
- `dish_restaurant_rankings_long.csv` — long-form rankings across three methods.
- `ranking_method_comparison.csv` — method agreement/disagreement summary.
- `top_recommendations_by_method.csv` — report-ready top recommendations.
- `grounded_recommendation_explanations.csv` — evidence-backed explanations.

## Learning objectives

By completing this task, you should be able to:

1. Use a curated dish vocabulary from Task 3 to extract dish-local review evidence.
2. Compare ratings-only, lexicon-based, and transformer/fallback sentiment rankings.
3. Inspect evidence when ranking methods agree or disagree.
4. Produce a short explanation grounded in retrieved dish-local review snippets.


## 0. Runtime and package setup

### Local installation and run

You can download and run this notebook on your personal computer. Running locally is recommended if transformer sentiment is part of your evaluation — large model downloads are more stable on a local machine with persistent disk space.

[Jupyter Installation Instructions](https://docs.jupyter.org/en/latest/install.html)

### Google Colab

You can also run this notebook on Google Colab, though transformer model downloads can be slow and Colab may reset during long runs.

- Use **Runtime → Change runtime type → Python 3**.
- A GPU helps with transformer inference but is not required.

This notebook is designed to run in:

- `smoke_test` mode with no network access;
- local/manual dataset mode when Yelp files are already available;
- Colab/Kaggle mode when Kaggle access is configured.

The notebook does **not** require live network access in `smoke_test` mode. Classical ratings/lexicon methods are required baselines. Transformer sentiment is the required modern comparison path for real submissions; defensive fallback code may run in smoke mode without a downloaded model.


## *Smoke Test Mode*

*Use `smoke_test` mode only to verify that your notebook runs end-to-end when real data is unavailable. In smoke mode the synthetic fixture contains only a small number of dishes and reviews; dish-local evidence will be sparse and rankings will not be meaningful. Base your final report conclusions on the real Yelp dataset.*


In [ ]:
# Optional package installation for Colab.
# Uncomment only if needed in Colab.
# !pip -q install kagglehub transformers sentencepiece torch

import os
import re
import json
import math
import zipfile
import tarfile
import random
import shutil
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import precision_recall_fscore_support

In [ ]:
# Reproducibility and display defaults.
RANDOM_SEED = int(os.environ.get("DMC_RANDOM_SEED", "42"))
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 30)

# 中文注释：从任意启动目录自动定位项目根目录，避免输出误写到 notebooks/artifacts。
def _find_project_root_for_task4(start: Path) -> Path:
    # 中文注释：优先使用显式环境变量，便于本地、Colab 或其他机器复现。
    env_root = os.environ.get("DMC_PROJECT_ROOT", "").strip()
    # 中文注释：如果用户已经指定项目根目录，则直接使用该目录。
    if env_root:
        # 中文注释：展开用户目录并解析为绝对路径。
        return Path(env_root).expanduser().resolve()
    # 中文注释：从当前目录逐级向上寻找项目根目录。
    for candidate in [start, *start.parents]:
        # 中文注释：项目根目录应包含 notebooks 目录。
        has_notebooks = (candidate / "notebooks").is_dir()
        # 中文注释：真实数据、Task 3 输出或项目包目录用于确认项目根目录。
        has_root_marker = (
            (candidate / "data" / "raw").is_dir()
            or (candidate / "artifacts" / "task3" / "final_dish_list.csv").is_file()
            or (candidate / "cs598capstone").is_dir()
        )
        # 中文注释：满足项目结构信号时返回候选根目录。
        if has_notebooks and has_root_marker:
            # 中文注释：返回规范化后的项目根目录。
            return candidate.resolve()
    # 中文注释：如果没有识别到项目根目录，则退回当前启动目录。
    return start.resolve()

# 中文注释：先定位项目根目录，再创建输出目录。
PROJECT_ROOT = _find_project_root_for_task4(Path.cwd().resolve())
# 中文注释：切换到项目根目录，保证后续相对路径都从项目根目录解析。
os.chdir(PROJECT_ROOT)
# 中文注释：Task 4 的所有输出统一写入项目根目录下 artifacts/task4。
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "task4"
# 中文注释：确保 Task 4 输出目录存在。
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Artifact directory:", ARTIFACT_DIR.resolve())

In [ ]:
# Xiaofei's code — Task 4 local path bootstrap / 本地真实数据与输出路径配置
# 中文注释：这个 cell 要在 Section 1 Configuration 之前运行，用来避免最终分析误用 smoke_test。

# 中文注释：重新导入 os，确保单独运行本 cell 时也能设置环境变量。
import os
# 中文注释：重新导入 Path，确保单独运行本 cell 时也能处理路径。
from pathlib import Path

# 中文注释：定义项目根目录查找函数，避免从 notebooks 子目录运行时输出到错误位置。
def find_project_root(start: Path) -> Path:
    # 中文注释：从当前目录开始逐级向上检查候选目录。
    for candidate in [start, *start.parents]:
        # 中文注释：项目根目录应包含 notebooks 目录。
        has_notebooks = (candidate / "notebooks").is_dir()
        # 中文注释：最终分析必须使用真实 Yelp 数据目录。
        has_raw_data = (candidate / "data" / "raw").is_dir()
        # 中文注释：Task 4 需要读取 Task 3 生成的最终菜品表。
        has_task3_artifact = (candidate / "artifacts" / "task3" / "final_dish_list.csv").is_file()
        # 中文注释：本项目通常包含本地虚拟环境或项目包目录。
        has_project_package = (candidate / "cs598capstone").is_dir()
        # 中文注释：同时具备 notebooks 且具备真实数据、Task 3 输出或项目包时，认为是项目根目录。
        if has_notebooks and (has_raw_data or has_task3_artifact or has_project_package):
            # 中文注释：返回识别出的项目根目录。
            return candidate
    # 中文注释：如果没有找到严格匹配的根目录，则退回到启动目录。
    return start

# 中文注释：记录前面初始化 cell 可能已经创建的旧输出目录。
bootstrap_artifact_dir = globals().get("ARTIFACT_DIR")
# 中文注释：优先读取显式配置的项目根目录。
env_root = os.environ.get("DMC_PROJECT_ROOT", "").strip()
# 中文注释：如果用户配置了项目根目录，则使用该路径。
if env_root:
    # 中文注释：展开并规范化用户提供的项目根目录。
    PROJECT_ROOT = Path(env_root).expanduser().resolve()
# 中文注释：否则从当前工作目录自动向上查找项目根目录。
else:
    # 中文注释：根据当前目录自动识别项目根目录。
    PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# 中文注释：切换到项目根目录，保证所有相对路径都写入根目录下的 artifacts。
os.chdir(PROJECT_ROOT)

# 中文注释：强制使用真实本地 Yelp 数据，不使用 smoke_test。
os.environ["DMC_DATA_SOURCE"] = "manual"
# 中文注释：同步设置 Task 4 专用数据来源变量，避免旧环境变量回落到 smoke_test。
os.environ["TASK4_DATA_SOURCE"] = "manual"
# 中文注释：指定真实 Yelp JSON 数据目录。
os.environ["DMC_DATA_DIR"] = str(PROJECT_ROOT / "data" / "raw")
# 中文注释：同步设置 Task 4 专用数据目录变量。
os.environ["TASK4_DATA_DIR"] = str(PROJECT_ROOT / "data" / "raw")
# 中文注释：指定 Task 3 传给 Task 4 的最终菜品表路径。
os.environ["TASK4_DISH_LIST_PATH"] = str(PROJECT_ROOT / "artifacts" / "task3" / "final_dish_list.csv")
# 中文注释：指定 Task 4 使用的真实 RoBERTa sentiment 模型 ID。
os.environ["TASK4_TRANSFORMER_MODEL_ID"] = "cardiffnlp/twitter-roberta-base-sentiment"
# 中文注释：提高真实 transformer 推理上限，覆盖当前真实数据中的全部菜品证据。
os.environ["TASK4_MAX_TRANSFORMER_EVIDENCE_ROWS"] = "12000"
# 中文注释：定义本地 RoBERTa sentiment 模型目录。
LOCAL_TRANSFORMER_MODEL_DIR = PROJECT_ROOT / "models" / "cardiffnlp-twitter-roberta-base-sentiment"
# 中文注释：如果本地模型目录已经存在，则优先使用本地模型，避免运行时再次下载。
if LOCAL_TRANSFORMER_MODEL_DIR.exists():
    # 中文注释：把本地 RoBERTa 模型路径写入 Task 4 配置。
    os.environ["TASK4_TRANSFORMER_MODEL_PATH"] = str(LOCAL_TRANSFORMER_MODEL_DIR)
    # 中文注释：本地模型存在时关闭下载，保证结果可复现。
    os.environ["TASK4_ALLOW_TRANSFORMER_DOWNLOAD"] = "0"
# 中文注释：如果本地模型目录不存在，则允许从 Hugging Face 下载真实 transformer 模型。
else:
    # 中文注释：清空本地模型路径，后续配置会走模型 ID 下载路径。
    os.environ["TASK4_TRANSFORMER_MODEL_PATH"] = ""
    # 中文注释：允许下载真实 RoBERTa sentiment 模型，避免回退到 fallback。
    os.environ["TASK4_ALLOW_TRANSFORMER_DOWNLOAD"] = "1"
# 中文注释：指定 Task 4 的所有输出 artifact 目录。
TASK4_OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "task4"
# 中文注释：把输出目录也写入环境变量，便于后续检查和复现。
os.environ["TASK4_ARTIFACT_DIR"] = str(TASK4_OUTPUT_DIR)
# 中文注释：重设 notebook 全局输出目录，覆盖前面可能在 notebooks 子目录下创建的相对路径。
ARTIFACT_DIR = TASK4_OUTPUT_DIR
# 中文注释：确保 Task 4 输出目录存在。
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 中文注释：如果前面 cell 创建了错误位置的空输出目录，则安全清理空目录。
if bootstrap_artifact_dir is not None:
    # 中文注释：把旧输出目录转换为 Path 对象。
    bootstrap_artifact_dir = Path(bootstrap_artifact_dir)
    # 中文注释：只有旧目录和真实输出目录不一致时才尝试清理。
    if bootstrap_artifact_dir.resolve() != ARTIFACT_DIR.resolve():
        # 中文注释：尝试删除错误位置下的空 task4 目录。
        try:
            # 中文注释：只删除空目录，目录内有文件时会自动保留。
            bootstrap_artifact_dir.rmdir()
        # 中文注释：目录不存在或非空时不影响 notebook 继续运行。
        except OSError:
            # 中文注释：跳过无法安全删除的旧目录。
            pass
        # 中文注释：尝试删除错误位置下的空 artifacts 父目录。
        try:
            # 中文注释：只删除空父目录，避免误删已有输出。
            bootstrap_artifact_dir.parent.rmdir()
        # 中文注释：父目录不存在或非空时不影响 notebook 继续运行。
        except OSError:
            # 中文注释：跳过无法安全删除的旧父目录。
            pass

# 中文注释：列出最终分析必须存在的真实 Yelp 数据文件。
required_yelp_files = [PROJECT_ROOT / "data" / "raw" / "yelp_academic_dataset_business.json", PROJECT_ROOT / "data" / "raw" / "yelp_academic_dataset_review.json"]
# 中文注释：检查真实 Yelp 数据文件是否缺失。
missing_yelp_files = [path for path in required_yelp_files if not path.exists()]
# 中文注释：如果缺少真实数据文件，则立即报错，避免回退到 smoke_test。
if missing_yelp_files:
    # 中文注释：抛出包含缺失文件路径的错误。
    raise FileNotFoundError(f"Task 4 requires real Yelp data files, missing: {missing_yelp_files}")
# 中文注释：如果数据来源仍然是 smoke_test，则立即报错。
if os.environ.get("DMC_DATA_SOURCE") == "smoke_test" or os.environ.get("TASK4_DATA_SOURCE") == "smoke_test":
    # 中文注释：禁止最终分析使用 smoke_test。
    raise ValueError("Task 4 final analysis must use manual data, not smoke_test.")
# 中文注释：如果本地模型路径已设置，则检查关键模型文件是否存在。
if os.environ.get("TASK4_TRANSFORMER_MODEL_PATH"):
    # 中文注释：定义 PyTorch RoBERTa 权重文件路径。
    transformer_weight_path = Path(os.environ["TASK4_TRANSFORMER_MODEL_PATH"]) / "pytorch_model.bin"
    # 中文注释：定义 RoBERTa tokenizer 词表文件路径。
    transformer_vocab_path = Path(os.environ["TASK4_TRANSFORMER_MODEL_PATH"]) / "vocab.json"
    # 中文注释：本地模型缺少关键文件时立即报错，避免静默回退到 fallback。
    if not transformer_weight_path.exists() or not transformer_vocab_path.exists():
        # 中文注释：抛出包含本地模型目录的错误信息。
        raise FileNotFoundError(f"Local transformer model is incomplete: {os.environ['TASK4_TRANSFORMER_MODEL_PATH']}")

# 中文注释：打印项目根目录，便于检查路径是否正确。
print("PROJECT_ROOT:", PROJECT_ROOT)
# 中文注释：打印当前工作目录，确认已经切换到项目根目录。
print("cwd:", os.getcwd())
# 中文注释：打印真实数据来源配置。
print("DATA_SOURCE will be:", os.environ.get("DMC_DATA_SOURCE"))
# 中文注释：打印真实数据目录配置。
print("DMC_DATA_DIR will be:", os.environ.get("DMC_DATA_DIR"))
# 中文注释：打印 Task 3 菜品表路径配置。
print("TASK4_DISH_LIST_PATH:", os.environ.get("TASK4_DISH_LIST_PATH"))
# 中文注释：打印真实 transformer 本地模型路径配置。
print("TASK4_TRANSFORMER_MODEL_PATH:", os.environ.get("TASK4_TRANSFORMER_MODEL_PATH") or "download_enabled")
# 中文注释：打印是否允许下载真实 transformer 模型。
print("TASK4_ALLOW_TRANSFORMER_DOWNLOAD:", os.environ.get("TASK4_ALLOW_TRANSFORMER_DOWNLOAD"))
# 中文注释：打印 Task 4 输出目录配置。
print("ARTIFACT_DIR:", ARTIFACT_DIR.resolve())


## 1. Configuration

Use `smoke_test` mode to check that the notebook executes without network access. For final analysis, use the assigned real or manual Yelp data path and the `final_dish_list.csv` produced in Task 3 when available. Keep the sampling and transformer limits bounded so the notebook remains practical in Colab or local course environments.

## About RoBERTa Sentiment Classification

RoBERTa sentiment classification uses a pretrained transformer model to assign sentiment labels or scores to text. In Task 4, it is the modern comparison method for scoring dish-local evidence sentences or windows. The goal is not to assume the transformer is always better; it is to compare its rankings with ratings-only and lexicon sentiment outputs and explain where the methods agree or disagree.

Transformer sentiment can capture broader context than a simple lexicon, but it also introduces runtime, dependency, model-selection, and reproducibility costs. If the model is unavailable, the notebook must use a clearly labeled fallback and avoid describing the fallback as equivalent. You should report bounded inference settings and inspect qualitative examples, especially where transformer/fallback scores change restaurant rankings.

**Key Resources:**
- [RoBERTa paper](https://arxiv.org/abs/1907.11692) - Foundational RoBERTa model paper.
- [Hugging Face Transformers Pipelines](https://huggingface.co/docs/transformers/main_classes/pipelines) - Practical sentiment pipeline documentation.
- [Hugging Face model hub](https://huggingface.co/models?pipeline_tag=text-classification) - Text-classification model catalog.

In [ ]:
# Dataset source options:
# - "smoke_test": create a tiny synthetic Yelp-shaped dataset; no network required.
# - "manual": use DATA_DIR pointing to local Yelp JSON files.
# - "kaggle_latest": use kagglehub to download the latest public Yelp dataset.
DATA_SOURCE = os.environ.get("DMC_DATA_SOURCE", os.environ.get("TASK4_DATA_SOURCE", "smoke_test"))
DATA_DIR = os.environ.get("DMC_DATA_DIR", os.environ.get("TASK4_DATA_DIR", ""))

TARGET_CATEGORY = "Chinese"
REQUIRE_RESTAURANTS = True

# Task 3 handoff. This is the preferred source of dish vocabulary.
TASK3_DISH_LIST_PATH = Path(os.environ.get(
    "TASK4_DISH_LIST_PATH",
    Path("artifacts") / "task3" / "final_dish_list.csv"
))

# Runtime bounds. Do not load the entire Yelp review file into memory.
MAX_TARGET_BUSINESSES = int(os.environ.get("TASK4_MAX_TARGET_BUSINESSES", "80"))
MAX_REVIEWS_TOTAL = int(os.environ.get("TASK4_MAX_REVIEWS_TOTAL", "25000"))
MAX_REVIEWS_PER_BUSINESS = int(os.environ.get("TASK4_MAX_REVIEWS_PER_BUSINESS", "400"))

# Business sampling. The default selects from all eligible target-category businesses using
# category focus and review count, rather than taking the first businesses encountered in the file.
TARGET_BUSINESS_SELECTION_STRATEGY = os.environ.get("TASK4_BUSINESS_SELECTION_STRATEGY", "quality_weighted_review_count")
MIN_BUSINESS_REVIEW_COUNT_FOR_SELECTION = int(os.environ.get("TASK4_MIN_BUSINESS_REVIEW_COUNT_FOR_SELECTION", "5"))
MAX_COMPETING_CUISINE_TAGS = int(os.environ.get("TASK4_MAX_COMPETING_CUISINE_TAGS", "2"))
USE_DISH_ALIASES = os.environ.get("TASK4_USE_DISH_ALIASES", "1") == "1"

# Mention extraction / ranking parameters.
LOCAL_CONTEXT_SENTENCES = int(os.environ.get("TASK4_LOCAL_CONTEXT_SENTENCES", "1"))
MIN_MENTIONS_FOR_RANKING = int(os.environ.get("TASK4_MIN_MENTIONS_FOR_RANKING", "2"))
MODERATE_DISH_MENTIONS = int(os.environ.get("TASK4_MODERATE_DISH_MENTIONS", "10"))
MODERATE_DISH_BUSINESSES = int(os.environ.get("TASK4_MODERATE_DISH_BUSINESSES", "3"))
STRONG_DISH_MENTIONS = int(os.environ.get("TASK4_STRONG_DISH_MENTIONS", "50"))
STRONG_DISH_BUSINESSES = int(os.environ.get("TASK4_STRONG_DISH_BUSINESSES", "10"))
MIN_BUSINESS_MENTIONS_FOR_RANKING = int(os.environ.get("TASK4_MIN_BUSINESS_MENTIONS_FOR_RANKING", "2"))
TOP_K_PER_DISH = int(os.environ.get("TASK4_TOP_K_PER_DISH", "8"))
SELECTED_DISHES_FOR_PLOTS = [d.strip() for d in os.environ.get(
    "TASK4_SELECTED_DISHES",
    "dumplings,kung pao chicken,mapo tofu,beef noodle soup,hot pot"
).split(",") if d.strip()]

# Transformer sentiment configuration.
# By default, we avoid model downloads. If a local model path is available, set TASK4_TRANSFORMER_MODEL_PATH.
USE_TRANSFORMER_SENTIMENT = os.environ.get("TASK4_USE_TRANSFORMER_SENTIMENT", "1") == "1"
TRANSFORMER_MODEL_PATH = os.environ.get("TASK4_TRANSFORMER_MODEL_PATH", "").strip()
ALLOW_TRANSFORMER_DOWNLOAD = os.environ.get("TASK4_ALLOW_TRANSFORMER_DOWNLOAD", "0") == "1"
MAX_TRANSFORMER_EVIDENCE_ROWS = int(os.environ.get("TASK4_MAX_TRANSFORMER_EVIDENCE_ROWS", "2500"))
TRANSFORMER_MODEL_ID = os.environ.get(
    "TASK4_TRANSFORMER_MODEL_ID",
    "cardiffnlp/twitter-roberta-base-sentiment"
)

print({
    "DATA_SOURCE": DATA_SOURCE,
    "DATA_DIR": DATA_DIR,
    "TARGET_CATEGORY": TARGET_CATEGORY,
    "TASK3_DISH_LIST_PATH": str(TASK3_DISH_LIST_PATH),
    "MAX_TARGET_BUSINESSES": MAX_TARGET_BUSINESSES,
    "MAX_REVIEWS_TOTAL": MAX_REVIEWS_TOTAL,
    "MAX_REVIEWS_PER_BUSINESS": MAX_REVIEWS_PER_BUSINESS,
    "TARGET_BUSINESS_SELECTION_STRATEGY": TARGET_BUSINESS_SELECTION_STRATEGY,
    "MIN_BUSINESS_REVIEW_COUNT_FOR_SELECTION": MIN_BUSINESS_REVIEW_COUNT_FOR_SELECTION,
    "MAX_COMPETING_CUISINE_TAGS": MAX_COMPETING_CUISINE_TAGS,
    "USE_DISH_ALIASES": USE_DISH_ALIASES,
    "MODERATE_DISH_MENTIONS": MODERATE_DISH_MENTIONS,
    "MODERATE_DISH_BUSINESSES": MODERATE_DISH_BUSINESSES,
    "STRONG_DISH_MENTIONS": STRONG_DISH_MENTIONS,
    "STRONG_DISH_BUSINESSES": STRONG_DISH_BUSINESSES,
    "MIN_BUSINESS_MENTIONS_FOR_RANKING": MIN_BUSINESS_MENTIONS_FOR_RANKING,
    "USE_TRANSFORMER_SENTIMENT": USE_TRANSFORMER_SENTIMENT,
    "TRANSFORMER_MODEL_PATH": TRANSFORMER_MODEL_PATH or None,
    "ALLOW_TRANSFORMER_DOWNLOAD": ALLOW_TRANSFORMER_DOWNLOAD,
    "MAX_TRANSFORMER_EVIDENCE_ROWS": MAX_TRANSFORMER_EVIDENCE_ROWS,
})

## 2. Dataset acquisition and discovery

The loader searches recursively for Yelp-style `business` and `review` JSON files. If using Kaggle, the dataset may be downloaded as a nested archive, so the loader can extract common `.zip`, `.tar`, `.tar.gz`, and `.tgz` files.

There is intentionally no legacy course-dataset path in this notebook. The refreshed course uses the current public Yelp dataset or a local/manual copy of Yelp-shaped JSON files.

In [ ]:
def extract_archives_if_needed(dataset_dir: Path) -> Path:
    dataset_dir = Path(dataset_dir)
    for archive in list(dataset_dir.rglob("*")):
        name = archive.name.lower()
        try:
            if archive.is_file() and name.endswith(".zip"):
                out_dir = archive.with_suffix("")
                if not out_dir.exists():
                    print(f"Extracting {archive} -> {out_dir}")
                    out_dir.mkdir(parents=True, exist_ok=True)
                    with zipfile.ZipFile(archive, "r") as zf:
                        zf.extractall(out_dir)
            elif archive.is_file() and (name.endswith(".tar") or name.endswith(".tar.gz") or name.endswith(".tgz")):
                out_dir = archive.parent / re.sub(r"(\.tar\.gz|\.tgz|\.tar)$", "", archive.name, flags=re.I)
                if not out_dir.exists():
                    print(f"Extracting {archive} -> {out_dir}")
                    out_dir.mkdir(parents=True, exist_ok=True)
                    with tarfile.open(archive, "r:*") as tf:
                        tf.extractall(out_dir)
        except Exception as exc:
            warnings.warn(f"Could not extract {archive}: {exc}")
    return dataset_dir


def create_smoke_test_dataset(out_dir: Path, n_restaurants: int = 12, reviews_per_restaurant: int = 26) -> Path:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    business_path = out_dir / "yelp_academic_dataset_business.json"
    review_path = out_dir / "yelp_academic_dataset_review.json"

    restaurants = [
        ("b1", "Sichuan Garden", "Chinese, Szechuan, Restaurants", "Phoenix"),
        ("b2", "Golden Dumpling House", "Chinese, Dim Sum, Restaurants", "Phoenix"),
        ("b3", "Noodle Dynasty", "Chinese, Noodles, Restaurants", "Phoenix"),
        ("b4", "Dragon Palace", "Chinese, Cantonese, Restaurants", "Phoenix"),
        ("b5", "Hot Pot Village", "Chinese, Hot Pot, Restaurants", "Phoenix"),
        ("b6", "Jade Wok", "Chinese, Restaurants", "Phoenix"),
        ("b7", "Taco Vista", "Mexican, Restaurants", "Phoenix"),
        ("b8", "Pasta Corner", "Italian, Restaurants", "Phoenix"),
        ("b9", "Thai Basil", "Thai, Restaurants", "Phoenix"),
        ("b10", "Pho Street", "Vietnamese, Restaurants", "Phoenix"),
    ]

    with business_path.open("w", encoding="utf-8") as f:
        for bid, name, cats, city in restaurants:
            row = {
                "business_id": bid,
                "name": name,
                "categories": cats,
                "city": city,
                "state": "AZ",
                "stars": round(random.uniform(3.2, 4.8), 1),
                "review_count": reviews_per_restaurant,
            }
            f.write(json.dumps(row) + "\n")

    dish_patterns = {
        "b1": [
            ("mapo tofu", "The mapo tofu is spicy, numbing, and excellent.", 5),
            ("kung pao chicken", "The kung pao chicken tastes fresh and balanced.", 5),
            ("hot pot", "The hot pot broth was rich but sometimes too salty.", 4),
            ("dumplings", "The dumplings were overcooked and bland.", 2),
        ],
        "b2": [
            ("dumplings", "The dumplings are juicy, delicate, and absolutely delicious.", 5),
            ("scallion pancakes", "The scallion pancakes are crisp and tasty.", 5),
            ("beef noodle soup", "The beef noodle soup was fine but not memorable.", 3),
            ("mapo tofu", "The mapo tofu was watery and disappointing.", 2),
        ],
        "b3": [
            ("beef noodle soup", "The beef noodle soup has tender beef and a deep broth.", 5),
            ("dan dan noodles", "The dan dan noodles are flavorful and spicy.", 5),
            ("dumplings", "The dumplings were okay but not the reason to visit.", 3),
            ("hot pot", "The hot pot was expensive and weak.", 2),
        ],
        "b4": [
            ("peking duck", "The peking duck has crisp skin and excellent flavor.", 5),
            ("dumplings", "The dumplings are solid and comforting.", 4),
            ("scallion pancakes", "The scallion pancakes arrived cold.", 2),
        ],
        "b5": [
            ("hot pot", "The hot pot is fun, generous, and perfect for groups.", 5),
            ("mapo tofu", "The mapo tofu is spicy but a little oily.", 4),
            ("beef noodle soup", "The beef noodle soup was bland.", 2),
        ],
        "b6": [
            ("kung pao chicken", "The kung pao chicken is sweet, sticky, and mediocre.", 2),
            ("dumplings", "The dumplings are surprisingly good.", 4),
            ("beef noodle soup", "The beef noodle soup is average.", 3),
        ],
    }

    generic_positive = [
        "Service was friendly and the restaurant felt welcoming.",
        "The meal was worth the price and we would return.",
        "The kitchen was quick and the staff were attentive.",
    ]
    generic_negative = [
        "The service was slow and the dining room was noisy.",
        "The wait was frustrating and the table was sticky.",
        "The experience felt rushed and uneven.",
    ]

    rid = 0
    with review_path.open("w", encoding="utf-8") as f:
        for bid, name, cats, city in restaurants:
            for j in range(reviews_per_restaurant):
                rid += 1
                if bid in dish_patterns:
                    dish, sentence, stars = random.choice(dish_patterns[bid])
                    # Mix dish-local signal with generic whole-review language so sentence-level extraction matters.
                    if stars >= 4:
                        text = f"{sentence} {random.choice(generic_negative if random.random() < 0.25 else generic_positive)}"
                    elif stars <= 2:
                        text = f"{sentence} {random.choice(generic_positive if random.random() < 0.35 else generic_negative)}"
                    else:
                        text = f"{sentence} Service and price were acceptable."
                else:
                    stars = random.choice([3, 4, 5])
                    text = f"This is a decent non-Chinese restaurant with friendly staff and a good meal."
                row = {
                    "review_id": f"r{rid}",
                    "business_id": bid,
                    "user_id": f"u{random.randint(1,50)}",
                    "stars": stars,
                    "date": f"2021-{(j % 12) + 1:02d}-{(j % 28) + 1:02d}",
                    "text": text,
                }
                f.write(json.dumps(row) + "\n")

    # Also create a Task 3-style dish list for smoke mode.
    task3_dir = Path("artifacts") / "task3"
    task3_dir.mkdir(parents=True, exist_ok=True)
    dish_list_path = task3_dir / "final_dish_list.csv"
    if not dish_list_path.exists():
        pd.DataFrame({
            "dish": [
                "dumplings", "kung pao chicken", "mapo tofu", "beef noodle soup",
                "hot pot", "dan dan noodles", "peking duck", "scallion pancakes"
            ],
            "cuisine_or_category": ["Chinese"] * 8,
            "source_method": ["smoke_seed"] * 8,
            "curation_status": ["approved"] * 8,
            "notes": ["smoke-test approved dish"] * 8,
        }).to_csv(dish_list_path, index=False)
        print("Created smoke-test Task 3 dish list:", dish_list_path)

    return out_dir


def download_or_locate_dataset(data_source: str, data_dir: str = "") -> Path:
    data_source = data_source.strip().lower()
    if data_source == "smoke_test":
        smoke_dir = Path(os.environ.get("TASK4_SMOKE_DATA_DIR", Path("data") / "smoke_task4"))
        return create_smoke_test_dataset(smoke_dir)
    if data_source == "manual":
        if not data_dir:
            raise ValueError("DATA_SOURCE='manual' requires DATA_DIR to point to a local Yelp dataset directory.")
        path = Path(data_dir)
        if not path.exists():
            raise FileNotFoundError(f"Manual DATA_DIR does not exist: {path}")
        return extract_archives_if_needed(path)
    if data_source == "kaggle_latest":
        try:
            import kagglehub
        except ImportError as exc:
            raise ImportError(
                "kagglehub is not installed. In Colab, run: !pip install kagglehub"
            ) from exc
        path = Path(kagglehub.dataset_download("yelp-dataset/yelp-dataset"))
        return extract_archives_if_needed(path)
    raise ValueError(f"Unknown DATA_SOURCE: {data_source}")


def find_yelp_json_files(dataset_dir: Path):
    dataset_dir = Path(dataset_dir)
    json_files = list(dataset_dir.rglob("*.json"))
    business_candidates = [p for p in json_files if "business" in p.name.lower()]
    review_candidates = [p for p in json_files if "review" in p.name.lower()]
    if not business_candidates or not review_candidates:
        raise FileNotFoundError(
            f"Could not find Yelp business/review JSON files under {dataset_dir}. "
            "Expected filenames containing 'business' and 'review'."
        )
    business_path = sorted(business_candidates, key=lambda p: len(str(p)))[0]
    review_path = sorted(review_candidates, key=lambda p: len(str(p)))[0]
    return business_path, review_path


dataset_dir = download_or_locate_dataset(DATA_SOURCE, DATA_DIR)
business_path, review_path = find_yelp_json_files(dataset_dir)

print("Dataset directory:", dataset_dir)
print("Business file:", business_path)
print("Review file:", review_path)

## 3. Load the Task 3 dish list

The preferred input is `artifacts/task3/final_dish_list.csv`, created by Task 3. That file should contain only approved Task-4-ready dish/entity phrases.

For setup validation and smoke tests, this notebook can create a small fallback dish list. In the student-facing version, you should use their own Task 3 output.

In [ ]:
def load_task3_dish_list(path: Path, target_category: str) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        print(f"Task 3 dish list not found at {path}; creating instructor fallback list.")
        path.parent.mkdir(parents=True, exist_ok=True)
        fallback = pd.DataFrame({
            "dish": [
                "dumplings", "kung pao chicken", "mapo tofu", "beef noodle soup",
                "hot pot", "dan dan noodles", "peking duck", "scallion pancakes"
            ],
            "cuisine_or_category": [target_category] * 8,
            "source_method": ["instructor_fallback"] * 8,
            "curation_status": ["approved"] * 8,
            "notes": ["fallback dish for instructor validation"] * 8,
        })
        fallback.to_csv(path, index=False)

    df = pd.read_csv(path)
    if "dish" not in df.columns:
        raise ValueError(f"{path} must contain a 'dish' column.")

    df["dish"] = df["dish"].astype(str).str.strip().str.lower()
    df = df[df["dish"].str.len() > 0].copy()

    if "curation_status" in df.columns:
        approved_statuses = {"approved", "approve", "yes", "task4_ready", "approved_seed", "approved_after_review", "human_approved"}
        legacy_statuses = {"approved_auto"}
        status_lower = df["curation_status"].astype(str).str.lower()
        if status_lower.isin(legacy_statuses).any():
            warnings.warn(
                "Task 3 status 'approved_auto' is deprecated. Use 'approved_after_review' "
                "to make clear that algorithmic candidates were human-reviewed before Task 4."
            )
        before = len(df)
        df = df[status_lower.isin(approved_statuses | legacy_statuses)].copy()
        if before and df.empty:
            raise ValueError(
                f"{path} has a curation_status column, but no rows matched approved statuses: {sorted(approved_statuses)}"
            )

    if "task4_ready" in df.columns:
        ready_mask = df["task4_ready"].astype(str).str.lower().isin(["true", "1", "yes"])
        if ready_mask.any():
            df = df[ready_mask].copy()

    if "cuisine_or_category" not in df.columns:
        df["cuisine_or_category"] = target_category

    if "source_method" not in df.columns:
        df["source_method"] = "unknown"

    df = df.drop_duplicates(subset=["dish"]).sort_values("dish").reset_index(drop=True)
    if df.empty:
        raise ValueError(f"No approved dishes were loaded from {path}. Check Task 3 curation statuses and Task 4 readiness.")
    return df


dish_list = load_task3_dish_list(TASK3_DISH_LIST_PATH, TARGET_CATEGORY)
dish_list.to_csv(ARTIFACT_DIR / "loaded_task3_dish_list.csv", index=False)

print("Loaded dishes:", len(dish_list))
display(dish_list.head(20))

### Dish-List Quality Checkpoint

Before extracting mentions, inspect whether the Task 3 dish list is specific enough for ranking. Broad phrases can create noisy evidence, while very rare phrases may not have enough mentions across restaurants. Your report should explain which dishes are ready for ranking and which are too sparse or ambiguous.

## 4. Build the cuisine-specific review corpus

Task 4 is only as good as its evidence pool. We first choose a coherent set of target-category restaurants, then stream reviews only for those businesses so the notebook remains bounded.

The sampling strategy is a modeling decision. A first-N scan can accidentally select a thin or oddly mixed subset because Yelp JSON order is not a research design. The course scaffold therefore builds all eligible target-category businesses, scores category focus, and selects a bounded high-evidence restaurant sample. This improves ranking stability while keeping category bleed visible as a diagnostic.

### 4.1 Parse categories and score category focus

This block defines the category parser, restaurant gate, and focus score used for reproducible business selection. The score is not meant to be a hidden truth label; it is a transparent heuristic students can critique and adjust.

In [ ]:
def normalize_categories(catstr) -> list[str]:
    if catstr is None:
        return []
    if isinstance(catstr, list):
        raw = catstr
    else:
        raw = str(catstr).split(",")
    return [c.strip() for c in raw if c and str(c).strip()]


def category_match(categories: list[str], target: str) -> bool:
    target_l = target.lower()
    return any(c.lower() == target_l for c in categories)


def restaurant_gate(categories: list[str]) -> bool:
    if not REQUIRE_RESTAURANTS:
        return True
    lower = {c.lower() for c in categories}
    return "restaurants" in lower or "food" in lower


def iter_json_lines(path: Path, limit: int | None = None):
    with Path(path).open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue


COMPETING_CUISINE_TAGS = {
    "thai", "vietnamese", "japanese", "sushi bars", "korean", "malaysian", "cambodian",
    "filipino", "indian", "mexican", "italian", "ramen", "steakhouses",
    "american (new)", "american (traditional)",
}
TARGET_SUPPORT_TAGS = {
    "dim sum", "shanghainese", "szechuan", "cantonese", "taiwanese", "hong kong style cafe",
    "hot pot", "noodles", "soup", "seafood", "barbeque",
}
BROAD_VENUE_TAGS = {
    "shopping", "department stores", "public markets", "farmers market", "shopping centers",
    "sporting goods", "fashion", "dinner theater", "wineries", "local flavor", "grocery",
}


def category_focus_features(categories: list[str], target_category: str) -> dict:
    lower = {c.lower() for c in categories}
    competing = sorted(lower & COMPETING_CUISINE_TAGS)
    support = sorted(lower & TARGET_SUPPORT_TAGS)
    broad = sorted(lower & BROAD_VENUE_TAGS)
    score = 0.0
    if target_category.lower() in lower:
        score += 3.0
    if "restaurants" in lower:
        score += 1.0
    score += min(2.0, 0.4 * len(support))
    score -= 0.7 * len(competing)
    score -= 1.5 * len(broad)
    score -= 0.1 * max(0, len(lower) - 10)
    return {
        "category_focus_score": round(score, 3),
        "competing_cuisine_tag_count": len(competing),
        "supporting_target_tag_count": len(support),
        "broad_venue_tag_count": len(broad),
        "competing_cuisine_tags": "; ".join(competing),
        "supporting_target_tags": "; ".join(support),
        "broad_venue_tags": "; ".join(broad),
    }

### 4.2 Select target-category businesses

Here we create the candidate table and choose the bounded restaurant sample. Saving both `target_business_candidates.csv` and `target_business_selection.csv` makes the sampling decision auditable downstream.

In [ ]:
def collect_target_business_candidates(business_path: Path, target_category: str) -> pd.DataFrame:
    rows = []
    for row in iter_json_lines(business_path):
        cats = normalize_categories(row.get("categories"))
        if not (category_match(cats, target_category) and restaurant_gate(cats)):
            continue
        bid = row.get("business_id")
        if not bid:
            continue
        features = category_focus_features(cats, target_category)
        rows.append({
            "business_id": bid,
            "name": row.get("name", ""),
            "city": row.get("city", ""),
            "state": row.get("state", ""),
            "categories": ", ".join(cats),
            "business_stars": row.get("stars", np.nan),
            "business_review_count": row.get("review_count", np.nan),
            **features,
        })
    return pd.DataFrame(rows)


def select_target_businesses(candidates: pd.DataFrame) -> dict:
    if candidates.empty:
        return {}
    df = candidates.copy()
    df["business_review_count"] = pd.to_numeric(df["business_review_count"], errors="coerce").fillna(0)
    df["business_stars"] = pd.to_numeric(df["business_stars"], errors="coerce").fillna(0)

    if DATA_SOURCE != "smoke_test":
        df = df[df["business_review_count"] >= MIN_BUSINESS_REVIEW_COUNT_FOR_SELECTION].copy()
        df = df[df["competing_cuisine_tag_count"] <= MAX_COMPETING_CUISINE_TAGS].copy()
        if df.empty:
            warnings.warn("Business focus filters removed every target business; falling back to all candidates.")
            df = candidates.copy()

    if TARGET_BUSINESS_SELECTION_STRATEGY == "review_count":
        sort_cols = ["business_review_count", "business_stars", "name"]
        ascending = [False, False, True]
    elif TARGET_BUSINESS_SELECTION_STRATEGY == "random":
        df = df.sample(frac=1.0, random_state=RANDOM_SEED)
        sort_cols = ["name"]
        ascending = [True]
    else:
        sort_cols = ["category_focus_score", "business_review_count", "business_stars", "name"]
        ascending = [False, False, False, True]

    selected = df.sort_values(sort_cols, ascending=ascending).head(MAX_TARGET_BUSINESSES).copy()
    selected["selection_rank"] = range(1, len(selected) + 1)
    selected.to_csv(ARTIFACT_DIR / "target_business_selection.csv", index=False)
    return {
        row.business_id: {
            "business_id": row.business_id,
            "name": row.name,
            "city": row.city,
            "state": row.state,
            "categories": row.categories,
            "business_stars": row.business_stars,
            "business_review_count": row.business_review_count,
            "category_focus_score": row.category_focus_score,
            "competing_cuisine_tag_count": row.competing_cuisine_tag_count,
        }
        for row in selected.itertuples(index=False)
    }

target_business_candidates = collect_target_business_candidates(business_path, TARGET_CATEGORY)
target_business_candidates.to_csv(ARTIFACT_DIR / "target_business_candidates.csv", index=False)
target_businesses = select_target_businesses(target_business_candidates)

### 4.3 Stream reviews for the selected businesses

The review loader streams JSON lines and stops at explicit per-business and total-review limits. This keeps the notebook usable while giving each selected restaurant a fair chance to contribute evidence.

## About Ratings Baseline Aggregation

Ratings baseline aggregation ranks restaurants for each dish using review-star ratings from reviews that mention the dish. In Task 4, it is the transparent baseline for dish-to-restaurant ranking: students can ask whether restaurants with higher star ratings in dish-mention reviews also look promising for that dish. The method is simple, inspectable, and useful for detecting whether later sentiment methods are adding information.

The baseline has important limits. Review stars are assigned to the whole restaurant experience, not necessarily to the specific dish, and one review can mention several dishes. Sparse mention counts can also make a restaurant look artificially strong or weak. You should therefore report coverage fields such as mention count, review count, and number of businesses before trusting a rank.

**Key Resources:**
- [pandas GroupBy documentation](https://pandas.pydata.org/docs/reference/groupby.html) - Aggregation tools used for ranking tables.
- [scikit-learn DummyClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html) - Conceptual reference for why simple baselines matter.
- [Wikipedia: Recommender system](https://en.wikipedia.org/wiki/Recommender_system) - Background on rating-based recommendation ideas.

In [ ]:
def collect_reviews_for_businesses(review_path: Path, businesses: dict) -> pd.DataFrame:
    wanted = set(businesses)
    per_business_counts = Counter()
    rows = []
    for row in iter_json_lines(review_path):
        bid = row.get("business_id")
        if bid not in wanted:
            continue
        if per_business_counts[bid] >= MAX_REVIEWS_PER_BUSINESS:
            continue
        text = row.get("text", "")
        if not isinstance(text, str) or not text.strip():
            continue
        meta = businesses[bid]
        rows.append({
            "review_id": row.get("review_id", ""),
            "business_id": bid,
            "business_name": meta["name"],
            "city": meta["city"],
            "state": meta["state"],
            "categories": meta["categories"],
            "business_stars": meta["business_stars"],
            "business_review_count": meta.get("business_review_count", np.nan),
            "category_focus_score": meta.get("category_focus_score", np.nan),
            "competing_cuisine_tag_count": meta.get("competing_cuisine_tag_count", np.nan),
            "stars": float(row.get("stars", np.nan)),
            "date": row.get("date", ""),
            "text": text,
        })
        per_business_counts[bid] += 1
        if len(rows) >= MAX_REVIEWS_TOTAL:
            break
    return pd.DataFrame(rows)

reviews = collect_reviews_for_businesses(review_path, target_businesses)

if reviews.empty:
    raise ValueError(f"No reviews found for target category {TARGET_CATEGORY!r}. Try another category or increase bounds.")

reviews.to_csv(ARTIFACT_DIR / "task4_review_subset.csv", index=False)
reviews[["review_id", "business_id", "business_name", "stars", "date", "text"]].head().to_csv(
    ARTIFACT_DIR / "review_subset_preview.csv", index=False
)

print("Eligible target businesses:", len(target_business_candidates))
print("Selected target businesses:", len(target_businesses))
print("Collected reviews:", len(reviews))
display(pd.read_csv(ARTIFACT_DIR / "target_business_selection.csv").head(20))
display(reviews.groupby("business_name").agg(reviews=("review_id", "count"), avg_stars=("stars", "mean")).sort_values("reviews", ascending=False))

## 5. Dish mention extraction with local context

This is the most important modeling choice in Task 4. We do **dish-local sentiment** instead of whole-review sentiment because a restaurant review can praise one dish while criticizing another.

The extraction step has two responsibilities: improve recall, and make the resulting evidence auditable. Better recall is not automatically better ranking quality; aliases and broader dish names can attach one sentence to several dishes, so we explicitly diagnose that tradeoff.

### 5.1 Sentence windows and phrase normalization

We split reviews into sentence-like units and keep a small local window around each dish mention. This gives the sentiment models enough context while avoiding the main Task 4 failure mode: scoring the whole review as if every sentence were about the selected dish.

In [ ]:
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")
WORD_CHARS = r"A-Za-z0-9_'\-&"


def split_sentences(text: str) -> list[str]:
    text = re.sub(r"\s+", " ", str(text)).strip()
    if not text:
        return []
    parts = SENTENCE_SPLIT_RE.split(text)
    # If a review has no punctuation, fall back to chunks of reasonable length.
    if len(parts) == 1 and len(parts[0]) > 350:
        words = parts[0].split()
        parts = [" ".join(words[i:i+45]) for i in range(0, len(words), 45)]
    return [p.strip() for p in parts if p.strip()]


def _normalize_dish_phrase(value: str) -> str:
    return re.sub(r"\s+", " ", str(value).lower()).strip()

### 5.2 Curated dish aliases

Aliases handle spelling, romanization, and singular/plural variants such as `xiao long bao` versus `xlb`. They are deliberately curated and exported because every alias is a modeling assumption.

In [ ]:
CURATED_DISH_ALIASES = {
    "beef noodle soup": ["braised beef noodle soup", "taiwanese beef noodle soup"],
    "dan dan noodles": ["dan dan mian", "dandan noodles", "dandan mian"],
    "hot pot": ["hotpot"],
    "kung pao chicken": ["kung pow chicken"],
    "lo mein": ["lomein", "lo-mein"],
    "mapo tofu": ["ma po tofu", "mapo doufu", "ma po doufu"],
    "mongolian beef": ["mongolian-style beef"],
    "peking duck": ["roast duck", "roasted duck"],
    "pork buns": ["pork bun"],
    "scallion pancakes": ["scallion pancake", "green onion pancake", "green onion pancakes"],
    "shrimp dumplings": ["shrimp dumpling"],
    "soup dumplings": ["soup dumpling"],
    "spring rolls": ["spring roll"],
    "wonton soup": ["won ton soup", "wonton soups"],
    "xiao long bao": ["xlb", "xiaolongbao", "xiao-long-bao"],
}


def build_dish_alias_table(dish_list_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    canonical_dishes = sorted(dish_list_df["dish"].dropna().astype(str).map(_normalize_dish_phrase).unique())
    canonical_set = set(canonical_dishes)
    rows = []
    conflicts = []
    for dish in canonical_dishes:
        rows.append({
            "canonical_dish": dish,
            "match_phrase": dish,
            "match_type": "canonical",
            "alias_source": "task3_final_dish_list",
            "include_for_matching": True,
            "notes": "Canonical Task 3 approved dish.",
        })
        if USE_DISH_ALIASES:
            for alias in CURATED_DISH_ALIASES.get(dish, []):
                alias_norm = _normalize_dish_phrase(alias)
                if not alias_norm or alias_norm == dish:
                    continue
                conflict = alias_norm in canonical_set and alias_norm != dish
                if conflict:
                    conflicts.append({
                        "canonical_dish": dish,
                        "match_phrase": alias_norm,
                        "conflicting_canonical_dish": alias_norm,
                        "reason": "Alias is itself another canonical Task 3 dish.",
                    })
                    include = False
                else:
                    include = True
                rows.append({
                    "canonical_dish": dish,
                    "match_phrase": alias_norm,
                    "match_type": "alias",
                    "alias_source": "instructor_curated_task4_alias",
                    "include_for_matching": include,
                    "notes": "Curated variant for recall; inspect before using in another cuisine/category.",
                })
    alias_table = pd.DataFrame(rows).drop_duplicates(["canonical_dish", "match_phrase"]).reset_index(drop=True)
    conflict_table = pd.DataFrame(conflicts, columns=["canonical_dish", "match_phrase", "conflicting_canonical_dish", "reason"])
    alias_table.to_csv(ARTIFACT_DIR / "dish_alias_table.csv", index=False)
    conflict_table.to_csv(ARTIFACT_DIR / "dish_alias_conflicts.csv", index=False)
    return alias_table, conflict_table

### 5.3 Extract dish-local evidence rows

Each row below is one dish mention with the matched phrase, restaurant, review metadata, sentence, and local context window. These rows feed the ratings baseline, lexicon sentiment, transformer sentiment, and evidence-grounded explanations.


In [ ]:
def compile_dish_patterns(alias_table: pd.DataFrame) -> dict[str, list[tuple[str, str, re.Pattern]]]:
    patterns = defaultdict(list)
    usable = alias_table[alias_table["include_for_matching"].astype(bool)].copy()
    # Match longer phrases first so specific variants are easier to inspect.
    usable["phrase_len"] = usable["match_phrase"].astype(str).str.len()
    usable = usable.sort_values(["canonical_dish", "phrase_len"], ascending=[True, False])
    for row in usable.itertuples(index=False):
        phrase = _normalize_dish_phrase(row.match_phrase)
        escaped = re.escape(phrase)
        escaped = escaped.replace(r"\ ", r"\s+").replace(r"\-", r"[-\\s]+")
        pattern = rf"(?<![{WORD_CHARS}]){escaped}(?![{WORD_CHARS}])"
        patterns[row.canonical_dish].append((phrase, row.match_type, re.compile(pattern, flags=re.IGNORECASE)))
    return patterns


def extract_context(sentences: list[str], idx: int, radius: int = 1) -> str:
    start = max(0, idx - radius)
    end = min(len(sentences), idx + radius + 1)
    return " ".join(sentences[start:end])


def extract_dish_mentions(reviews: pd.DataFrame, dish_list: pd.DataFrame, alias_table: pd.DataFrame) -> pd.DataFrame:
    patterns = compile_dish_patterns(alias_table)
    evidence_rows = []

    for row in reviews.itertuples(index=False):
        sentences = split_sentences(row.text)
        if not sentences:
            continue
        for i, sent in enumerate(sentences):
            sent_lower = sent.lower()
            for dish, phrase_patterns in patterns.items():
                for match_phrase, match_type, pat in phrase_patterns:
                    if pat.search(sent_lower):
                        evidence_rows.append({
                            "dish": dish,
                            "matched_dish_phrase": match_phrase,
                            "dish_match_type": match_type,
                            "business_id": row.business_id,
                            "business_name": row.business_name,
                            "review_id": row.review_id,
                            "stars": row.stars,
                            "date": row.date,
                            "sentence": sent,
                            "context_window": extract_context(sentences, i, LOCAL_CONTEXT_SENTENCES),
                        })
                        break

    evidence = pd.DataFrame(evidence_rows)
    if not evidence.empty:
        evidence = evidence.drop_duplicates(subset=["dish", "business_id", "review_id", "sentence"]).reset_index(drop=True)
    return evidence

dish_alias_table, dish_alias_conflicts = build_dish_alias_table(dish_list)
display(dish_alias_table.head(40))
if not dish_alias_conflicts.empty:
    display(dish_alias_conflicts)

mention_evidence = extract_dish_mentions(reviews, dish_list, dish_alias_table)
mention_evidence.to_csv(ARTIFACT_DIR / "dish_mention_evidence.csv", index=False)

alias_coverage = (
    mention_evidence.groupby(["dish", "matched_dish_phrase", "dish_match_type"])
    .size()
    .reset_index(name="mention_count")
    .sort_values(["dish", "mention_count"], ascending=[True, False])
    if not mention_evidence.empty
    else pd.DataFrame(columns=["dish", "matched_dish_phrase", "dish_match_type", "mention_count"])
)
alias_coverage.to_csv(ARTIFACT_DIR / "dish_alias_coverage.csv", index=False)

search_cols = [
    "dish", "matched_dish_phrase", "dish_match_type", "business_name", "business_id",
    "review_id", "stars", "date", "sentence", "context_window",
]
if mention_evidence.empty:
    dish_search_index = pd.DataFrame(columns=["mention_id"] + search_cols + ["evidence_text"])
else:
    dish_search_index = mention_evidence.reset_index().rename(columns={"index": "mention_id"})
    dish_search_index["evidence_text"] = dish_search_index["context_window"].fillna(dish_search_index["sentence"])
    dish_search_index = dish_search_index[["mention_id"] + search_cols + ["evidence_text"]]
dish_search_index.to_csv(ARTIFACT_DIR / "dish_search_evidence_index.csv", index=False)

print("Dish mention evidence rows:", len(mention_evidence))
print("Alias-derived mention rows:", int((mention_evidence.get("dish_match_type", pd.Series(dtype=str)) == "alias").sum()) if not mention_evidence.empty else 0)
display(mention_evidence.head(20))
display(alias_coverage.head(40))

In [ ]:
# TODO (Category E — Guided): Compute dish coverage diagnostics.
# Estimated time: ~0.5 h
#
# Expected input: evidence DataFrame (rows: dish, business_id, text snippet, stars, ...)
# Expected output: coverage DataFrame with columns [dish, evidence_count, unique_businesses,
#   avg_stars_for_mentions, evidence_tier, main_analysis_ready]
#
# evidence_tier: "strong" if evidence_count >= EVIDENCE_TIER_STRONG,
#                "moderate" if >= EVIDENCE_TIER_MODERATE,
#                "weak" otherwise.
# main_analysis_ready: True if evidence_tier in ("strong", "moderate").
#
# Print: how many dishes have sufficient evidence vs. not.
# Display: coverage DataFrame sorted by evidence_count descending.

# TODO: groupby evidence["dish"] → aggregate evidence_count, unique_businesses, avg_stars_for_mentions
# 使用 mention_evidence 作为后续排名统一使用的证据表。
evidence = mention_evidence if "mention_evidence" in globals() else pd.DataFrame()
# 在没有证据时保留作业要求和后续可视化需要的列结构。
if evidence.empty:
    # 初始化空覆盖表，避免后续 cell 因缺列失败。
    coverage = pd.DataFrame(columns=["dish", "evidence_count", "unique_businesses", "avg_stars_for_mentions", "evidence_tier", "main_analysis_ready", "mention_count", "num_businesses_with_mentions", "avg_stars"])
# 在有证据时按菜品聚合覆盖统计。
else:
    # 汇总每个菜品的提及数、覆盖商家数和提及评论平均星级。
    coverage = evidence.groupby("dish").agg(evidence_count=("review_id", "count"), unique_businesses=("business_id", "nunique"), avg_stars_for_mentions=("stars", "mean")).reset_index()

# TODO: add evidence_tier and main_analysis_ready columns
# 根据提及数和覆盖商家数判定证据层级。
def _dish_evidence_tier(row):
    # 同时满足强证据的提及数和商家数阈值时标记为 strong。
    if row["evidence_count"] >= STRONG_DISH_MENTIONS and row["unique_businesses"] >= STRONG_DISH_BUSINESSES:
        # 返回强证据层级。
        return "strong"
    # 满足中等证据阈值时标记为 moderate。
    if row["evidence_count"] >= MODERATE_DISH_MENTIONS and row["unique_businesses"] >= MODERATE_DISH_BUSINESSES:
        # 返回中等证据层级。
        return "moderate"
    # 其余菜品证据较弱。
    return "weak"
# 对每个菜品应用证据层级函数。
coverage["evidence_tier"] = coverage.apply(_dish_evidence_tier, axis=1) if not coverage.empty else coverage.get("evidence_tier", pd.Series(dtype=str))
# 将强/中等证据标记为主分析可用。
coverage["main_analysis_ready"] = coverage["evidence_tier"].isin(["strong", "moderate"])
# 为后续可视化保留旧版提及数列名。
coverage["mention_count"] = coverage["evidence_count"]
# 为后续可视化保留旧版覆盖商家数列名。
coverage["num_businesses_with_mentions"] = coverage["unique_businesses"]
# 为后续可视化保留平均星级的短列名。
coverage["avg_stars"] = coverage["avg_stars_for_mentions"]
# 按证据数量降序排列覆盖诊断表。
coverage = coverage.sort_values("evidence_count", ascending=False).reset_index(drop=True)
# 将覆盖诊断表保存为 Task 4 核心 artifact。
coverage.to_csv(ARTIFACT_DIR / "dish_mention_coverage.csv", index=False)
# 统计可进入主分析的菜品数量。
ready_count = int(coverage["main_analysis_ready"].sum())
# 统计证据不足的菜品数量。
not_ready_count = int((~coverage["main_analysis_ready"]).sum())
# 打印覆盖诊断摘要。
print(f"Coverage ready dishes: {ready_count}; weak-evidence dishes: {not_ready_count}")
# 展示按证据数量排序后的覆盖表。
display(coverage)

# --- Validation ---
assert isinstance(coverage, pd.DataFrame) and "dish" in coverage.columns
print(f"\u2713 Coverage: {coverage['main_analysis_ready'].sum()} dishes have sufficient evidence")


### Local Evidence Quality Checkpoint

Dish-local evidence is only useful if the context actually refers to the target dish. Inspect the coverage table and example sentences before trusting a ranking.


## Data Mining Concept: Dish Sentiment as a Ranking Problem — Three Assumptions About Evidence

**The prediction task:**

Given a dish (e.g., "margherita pizza") and a set of restaurants that serve it, rank restaurants by **how positively diners talk about that specific dish**.

This is **not** the same as ranking restaurants by overall review stars. A 4.5-star restaurant might serve mediocre pizza but excellent pasta. The challenge is to extract dish-specific sentiment from noisy, multi-topic review text.

**Three methods, three assumptions:**

### Method A: Ratings-only baseline (Section 6)

**Assumption**: Reviews that *mention* a dish reflect sentiment about that dish. Use the review's overall star rating as a proxy for dish-specific sentiment.

**Strength**: Simple, requires no NLP, uses direct supervision signal (stars are human-labeled ground truth).

**Weakness**: 
- A 5-star review might say "the pizza was okay, but the service was amazing" — the rating reflects service, not pizza quality.
- Mentions can be incidental: "I usually order pizza, but today I tried the pasta" — this review tells you nothing about the pizza at *this* restaurant.
- Review-level stars are **coarse** (1-5 scale) and reviewer-dependent (some reviewers give 5 stars easily, others are strict).

### Method B: Lexicon-based sentiment (Section 7)

**Assumption**: Sentiment is detectable from **positive and negative word counts** in the local context around the dish mention (e.g., VADER, TextBlob, or a custom lexicon).

**Strength**: 
- Uses dish-local text (the sentence or ±N-word window containing the dish), not the entire review.
- Transparent: you can inspect which words triggered positive/negative scores.
- Fast: no model training, just dictionary lookups.

**Weakness**:
- Misses **context and qualification**: "The pizza was not bad" or "I expected better pizza" may score incorrectly.
- No semantic understanding: "the crust was burnt" is negative, but a basic lexicon might not know "burnt" applies to food quality.
- Domain mismatch: VADER was trained on social media, not restaurant reviews. "Sick" means "cool" on Twitter but "unwell" in a hygiene complaint.

### Method C: Transformer-based sentiment (Section 8)

**Assumption**: A **pre-trained neural model** (e.g., RoBERTa fine-tuned on sentiment datasets) can capture contextual sentiment better than word counts.

**Strength**:
- Handles **negation, sarcasm, and qualification**: "not great", "could have been better", "I've had worse" are modeled as sequences, not bag-of-words.
- Learns from large-scale sentiment corpora (e.g., Twitter sentiment, IMDB reviews) via transfer learning.
- Produces calibrated probabilities (positive/negative/neutral) instead of heuristic scores.

**Weakness**:
- **Black box**: hard to explain *why* the model scored a sentence as positive or negative.
- **Domain shift**: A model trained on movie reviews or tweets may not generalize perfectly to dish-specific restaurant contexts.
- **Computational cost**: Requires GPU or careful batching; slower than lexicon methods.
- **Robustness**: Can be fooled by ambiguous cases ("interesting pizza" — is that positive or neutral?).

**Pedagogical insight for graduate students:**

This is a **supervision-versus-representation trade-off**:
- Method A (ratings) uses **strong but coarse supervision** (human-labeled stars) with **no linguistic understanding**.
- Method B (lexicon) uses **weak linguistic heuristics** (word polarity) with **local context**.
- Method C (transformer) uses **learned contextual representations** (neural language model) with **transferred sentiment knowledge**.

**None of these is universally better.** The right method depends on:
- **Data availability**: If you have millions of reviews, Method C's complexity might pay off. If you have 50 reviews, Method A or B is safer.
- **Interpretability requirements**: If you need to *explain* why a restaurant ranked #1 for "pizza", Method B (lexicon) is easier to justify than Method C (neural net).
- **Deployment constraints**: If you need to re-rank 10,000 dishes nightly, Method A or B is faster than running a transformer on every sentence.

**What you should compare in their reports:**

1. **Agreement**: Do all three methods agree on the top-5 restaurants for a given dish? If not, which method is the outlier?
2. **Coverage**: Does Method A (ratings) have more data (every review has stars) than Method B/C (which require text parsing)?
3. **Error cases**: Find dishes where Method A ranks a restaurant highly (5-star reviews) but Method B/C rank it low (negative dish-local sentiment). Inspect the review text — what's happening?
4. **Interpretability**: For a top-ranked restaurant, can you explain *why* it ranked #1 using Method B (show the positive words) vs. Method C (model scores)?

**The comparison below** evaluates whether the added complexity of Methods B and C improves dish-ranking quality over the simple ratings baseline.

## 6. Method A — Ratings-only baseline

The ratings-only baseline ranks restaurants using review stars from reviews that mention the dish. This is simple and transparent, but it is not dish-specific sentiment. A review could mention a dish while the star rating reflects service, price, ambience, or another dish.

We use a small shrinkage adjustment and a per-restaurant minimum mention threshold so a restaurant with one lucky 5-star mention does not automatically dominate a restaurant with many strong mentions. Sparse evidence is still visible in the mention coverage artifacts, but the ranking tables focus on restaurant/dish pairs with enough local evidence to compare.

In [ ]:
# TODO (Category B — Classic data mining): Implement ratings-based restaurant ranking.
# Estimated time: ~2 h
#
# Implement the following functions:
#
# ── A) star_to_unit_score(stars) → float ─────────────────────────────────────
#   Convert star rating to unit score: (stars - 3.0) / 2.0 → range [-1, 1].
#   Return np.nan on conversion failure.
#
# ── B) shrink_score(mean_score, n, global_mean, k=5.0) → float ───────────────
#   Bayesian shrinkage: ((n * mean_score) + (k * global_mean)) / (n + k).
#   Shrinks small-sample means toward the global mean.
#
# ── C) mention_bonus(n, max_n, weight=0.10) → float ─────────────────────────
#   Popularity bonus proportional to log(1+n)/log(1+max_n), scaled by weight.
#   Returns 0.0 if max_n <= 1.
#
# ── D) build_ratings_ranking(evidence) → DataFrame ───────────────────────────
#   Input: evidence DataFrame with columns [dish, business_id, business_name, stars, ...]
#   Returns: DataFrame with columns [dish, business_id, business_name, method,
#            mention_count, rank_score, rank]
#   - method = "ratings_baseline"
#   - rank_score = shrink_score(unit_mean, n, global_mean) + mention_bonus(n, max_n)
#   - rank = rank within each dish (1 = best), ties broken by business_id
#   Hint: groupby ["dish","business_id","business_name"], apply star_to_unit_score per row,
#         compute global_mean first, then per-group shrinkage + bonus.

# 定义所有排名方法共享的输出列顺序。
RANKING_COLUMNS = ["dish", "business_id", "business_name", "method", "mention_count", "avg_stars", "rank_score", "rank"]

def star_to_unit_score(stars) -> float:
    # TODO: implement unit score conversion
    # 捕获无法转换为数字的星级输入。
    try:
        # 将星级值转换为浮点数。
        stars_float = float(stars)
    # 遇到类型或取值错误时进入缺失值路径。
    except (TypeError, ValueError):
        # 返回 np.nan 供后续过滤。
        return np.nan
    # 将 1 到 5 星评分映射到 -1 到 1。
    return (stars_float - 3.0) / 2.0

def shrink_score(mean_score: float, n: int, global_mean: float, k: float = 5.0) -> float:
    # TODO: implement Bayesian shrinkage
    # 对缺失均值或空样本返回缺失值。
    if pd.isna(mean_score) or pd.isna(global_mean) or n <= 0:
        # 缺少有效输入时无法计算收缩分数。
        return np.nan
    # 按贝叶斯收缩公式把小样本均值拉向全局均值。
    return ((n * mean_score) + (k * global_mean)) / (n + k)

def mention_bonus(n: int, max_n: int, weight: float = 0.10) -> float:
    # TODO: implement popularity bonus
    # 当最大提及数不足以形成比例时不加热度奖励。
    if max_n <= 1:
        # 返回零奖励以避免除零或夸大单次提及。
        return 0.0
    # 使用对数比例让提及数奖励递增但不过度支配情感分数。
    return weight * (math.log1p(n) / math.log1p(max_n))

def build_ratings_ranking(evidence: "pd.DataFrame") -> "pd.DataFrame":
    # TODO: groupby dish+business; apply scoring; rank within dish; return DataFrame
    # 没有证据时返回统一 schema 的空表。
    if evidence is None or evidence.empty:
        # 构造空排名表以保证后续拼接逻辑稳定。
        return pd.DataFrame(columns=RANKING_COLUMNS)
    # 复制证据表，避免中间计算污染原始列。
    work = evidence.copy()
    # 将每条证据的 review stars 映射到单位区间分数。
    work["unit_score"] = work["stars"].map(star_to_unit_score)
    # 删除无法转换评分的证据行。
    work = work.dropna(subset=["unit_score"])
    # 如果评分列全无效，则返回空排名表。
    if work.empty:
        # 构造空排名表以保证后续拼接逻辑稳定。
        return pd.DataFrame(columns=RANKING_COLUMNS)
    # 计算全局平均单位评分作为收缩先验。
    raw_global_mean = work["unit_score"].mean()
    # 缺失时使用中性先验零分。
    global_mean = 0.0 if pd.isna(raw_global_mean) else float(raw_global_mean)
    # 汇总每个菜品和餐厅组合的提及数、平均星级和单位评分均值。
    grouped = work.groupby(["dish", "business_id", "business_name"], dropna=False).agg(mention_count=("review_id", "count"), avg_stars=("stars", "mean"), mean_score=("unit_score", "mean")).reset_index()
    # 过滤掉提及数不足的餐厅菜品组合。
    grouped = grouped[grouped["mention_count"] >= MIN_BUSINESS_MENTIONS_FOR_RANKING].copy()
    # 如果过滤后为空，则返回统一 schema 的空表。
    if grouped.empty:
        # 构造空排名表以保证后续拼接逻辑稳定。
        return pd.DataFrame(columns=RANKING_COLUMNS)
    # 记录最大提及数用于计算热度奖励。
    max_mentions = int(grouped["mention_count"].max())
    # 计算收缩后的评分并叠加温和的提及数奖励。
    grouped["rank_score"] = grouped.apply(lambda row: shrink_score(row["mean_score"], int(row["mention_count"]), global_mean) + mention_bonus(int(row["mention_count"]), max_mentions), axis=1)
    # 标记当前排名方法名称。
    grouped["method"] = "ratings_baseline"
    # 按菜品、分数降序和 business_id 升序排序以稳定处理并列。
    grouped = grouped.sort_values(["dish", "rank_score", "business_id"], ascending=[True, False, True]).reset_index(drop=True)
    # 在每个菜品内部生成从 1 开始的名次。
    grouped["rank"] = grouped.groupby("dish").cumcount().add(1)
    # 返回统一列顺序的排名表。
    return grouped[RANKING_COLUMNS]

# Call it:
ratings_ranking = build_ratings_ranking(evidence)
# --- Validation ---
assert isinstance(ratings_ranking, pd.DataFrame) and not ratings_ranking.empty
assert "rank" in ratings_ranking.columns and "method" in ratings_ranking.columns
print(f"\u2713 Ratings ranking: {len(ratings_ranking)} rows, {ratings_ranking['dish'].nunique()} dishes")


## 7. Method B — Lexicon-based dish-local sentiment

Lexicon sentiment scores the sentence/window around the dish mention using positive and negative word lists. This is still simple and interpretable, but it can distinguish cases where the review rating is high while the dish sentence is negative, or vice versa.

The lexicon below is intentionally small and transparent for teaching. You may use VADER/TextBlob or extend the lexicon, but they should explain the choice.

## About VADER/TextBlob Lexicon Sentiment

Lexicon sentiment methods score text by matching words and phrases against dictionaries of positive, negative, negated, or intensified expressions. In Task 4, lexicon sentiment is applied to dish-local evidence windows rather than entire reviews, so students can ask whether the specific sentence around a dish sounds favorable or unfavorable. This creates a more targeted ranking signal than overall review stars while remaining interpretable.

Lexicon sentiment is useful because students can inspect why a sentence received a score. It also has predictable weaknesses: sarcasm, negation, domain-specific food language, and short context windows can mislead dictionary-based scoring. Comparing lexicon rankings with ratings-only and transformer/fallback rankings teaches students to evaluate sentiment methods as evidence sources, not final answers.

**Key Resources:**
- [VADER Sentiment paper](https://ojs.aaai.org/index.php/ICWSM/article/view/14550) - Original social-media lexicon sentiment paper.
- [TextBlob Documentation](https://textblob.readthedocs.io/en/dev/) - Common Python sentiment API reference.
- [NLTK VADER Documentation](https://www.nltk.org/api/nltk.sentiment.vader.html) - Practical VADER implementation reference.

In [ ]:
# TODO (Category B — Classic data mining): Implement lexicon-based sentiment ranking.
# Estimated time: ~2 h
#
# Given POSITIVE_WORDS and NEGATIVE_WORDS sets (defined below), implement:
#
# ── A) simple_lexicon_sentiment(text) → float ─────────────────────────────────
#   Returns score in [-1, 1]:
#   pos_count = number of tokens in POSITIVE_WORDS;  neg_count = NEGATIVE_WORDS
#   score = (pos_count - neg_count) / max(1, total_tokens)
#   Normalize final score to [-1, 1] by dividing by a reasonable scale factor.
#   Hint: tokenize with re.findall(r"[a-z]+", text.lower())
#
# ── B) build_lexicon_ranking(evidence) → DataFrame ───────────────────────────
#   Same structure as build_ratings_ranking output, but method = "lexicon_sentiment".
#   rank_score = mean(simple_lexicon_sentiment) per (dish, business).
#   Apply shrink_score and mention_bonus as in ratings ranking.
#
# You may also experiment with TextBlob polarity: TextBlob(text).sentiment.polarity.
# If you use TextBlob, import it inside the function with a try/except fallback.

POSITIVE_WORDS = {
    "amazing", "awesome", "best", "crisp", "crispy", "delicious", "excellent", "favorite",
    "flavorful", "fresh", "friendly", "generous", "good", "great", "juicy", "love", "loved",
    "perfect", "rich", "solid", "spicy", "tasty", "tender", "worth", "welcoming", "balanced",
    "comforting", "deep", "fun", "quick", "attentive", "return", "surprisingly",
}
NEGATIVE_WORDS = {
    "awful", "bad", "bland", "cold", "disappointing", "dry", "expensive", "frustrating",
    "greasy", "mediocre", "noisy", "overcooked", "rushed", "salty", "slow", "sticky",
    "terrible", "uneven", "underwhelming", "worst", "overrated",
}

def simple_lexicon_sentiment(text: str) -> float:
    # TODO: tokenize; count positive/negative tokens; return normalized score
    # 空文本按中性情感处理。
    if not isinstance(text, str) or not text.strip():
        # 返回中性分数。
        return 0.0
    # 使用简单英文词形提取作为透明的词典匹配 token。
    tokens = re.findall(r"[a-z]+", text.lower())
    # 没有可用 token 时按中性情感处理。
    if not tokens:
        # 返回中性分数。
        return 0.0
    # 统计正向词典命中的 token 数。
    pos_count = sum(token in POSITIVE_WORDS for token in tokens)
    # 统计负向词典命中的 token 数。
    neg_count = sum(token in NEGATIVE_WORDS for token in tokens)
    # 计算按文本长度归一化的原始情感差值。
    raw_score = (pos_count - neg_count) / max(1, len(tokens))
    # 使用温和缩放放大短句中的明确情感词信号。
    scaled_score = raw_score / 0.20
    # 将最终分数裁剪到 [-1, 1] 区间。
    return float(np.clip(scaled_score, -1.0, 1.0))

def build_lexicon_ranking(evidence: "pd.DataFrame") -> "pd.DataFrame":
    # TODO: compute per-review sentiment; groupby dish+business; apply shrinkage + bonus; rank
    # 没有证据时返回统一 schema 的空表。
    if evidence is None or evidence.empty:
        # 构造空排名表以保证后续拼接逻辑稳定。
        return pd.DataFrame(columns=RANKING_COLUMNS)
    # 优先使用局部上下文窗口，缺失时退回命中的单句。
    text_col = "context_window" if "context_window" in evidence.columns else "sentence"
    # 将词典情感分数写回证据表，供后续解释和 artifact 使用。
    evidence["lexicon_score"] = evidence[text_col].fillna("").map(simple_lexicon_sentiment)
    # 使用带词典分数的证据副本进行聚合。
    work = evidence.dropna(subset=["lexicon_score"]).copy()
    # 如果没有有效词典分数，则返回空排名表。
    if work.empty:
        # 构造空排名表以保证后续拼接逻辑稳定。
        return pd.DataFrame(columns=RANKING_COLUMNS)
    # 计算全局词典情感均值作为收缩先验。
    raw_global_mean = work["lexicon_score"].mean()
    # 缺失时使用中性先验零分。
    global_mean = 0.0 if pd.isna(raw_global_mean) else float(raw_global_mean)
    # 汇总每个菜品和餐厅组合的提及数、平均星级和词典情感均值。
    grouped = work.groupby(["dish", "business_id", "business_name"], dropna=False).agg(mention_count=("review_id", "count"), avg_stars=("stars", "mean"), mean_score=("lexicon_score", "mean")).reset_index()
    # 过滤掉提及数不足的餐厅菜品组合。
    grouped = grouped[grouped["mention_count"] >= MIN_BUSINESS_MENTIONS_FOR_RANKING].copy()
    # 如果过滤后为空，则返回统一 schema 的空表。
    if grouped.empty:
        # 构造空排名表以保证后续拼接逻辑稳定。
        return pd.DataFrame(columns=RANKING_COLUMNS)
    # 记录最大提及数用于计算热度奖励。
    max_mentions = int(grouped["mention_count"].max())
    # 计算收缩后的词典情感评分并叠加温和的提及数奖励。
    grouped["rank_score"] = grouped.apply(lambda row: shrink_score(row["mean_score"], int(row["mention_count"]), global_mean) + mention_bonus(int(row["mention_count"]), max_mentions), axis=1)
    # 标记当前排名方法名称。
    grouped["method"] = "lexicon_sentiment"
    # 按菜品、分数降序和 business_id 升序排序以稳定处理并列。
    grouped = grouped.sort_values(["dish", "rank_score", "business_id"], ascending=[True, False, True]).reset_index(drop=True)
    # 在每个菜品内部生成从 1 开始的名次。
    grouped["rank"] = grouped.groupby("dish").cumcount().add(1)
    # 返回统一列顺序的排名表。
    return grouped[RANKING_COLUMNS]

# Call it:
lexicon_ranking = build_lexicon_ranking(evidence)
# --- Validation ---
assert isinstance(lexicon_ranking, pd.DataFrame) and not lexicon_ranking.empty
print(f"\u2713 Lexicon ranking: {len(lexicon_ranking)} rows")


## 8. Method C — Transformer sentiment (RoBERTa), with honest fallback

The notebook uses **`cardiffnlp/twitter-roberta-base-sentiment`** as the default transformer sentiment model. This RoBERTa model was fine-tuned on ~58 M tweets, which makes it a reasonable match for Yelp review language: short, informal, opinionated sentences with food and experience vocabulary.

The model outputs three classes: **LABEL_0** (negative), **LABEL_1** (neutral), **LABEL_2** (positive). The `transformer_label_to_score` function maps these to the −1 … +1 scale used by the other methods.

**Why not DistilBERT SST-2?**  DistilBERT-SST-2 is a binary (positive/negative) model trained on movie reviews. It works, but it has no neutral class and misses the gentler negative/positive signals common in restaurant reviews. RoBERTa with a neutral class better handles mixed reviews.

A transformer sentiment model can capture more context than a lexicon, but it adds dependencies, runtime cost, and model-domain assumptions. The notebook tries to load a local transformer model if configured.

If no local model is available and downloading is disabled (`ALLOW_TRANSFORMER_DOWNLOAD=0`), the notebook falls back to a labeled linear combination of the lexicon score and the star-rating prior. The fallback is clearly tagged so students know it is *not* a true transformer signal.

A transformer sentiment model can capture more context than a lexicon, but it adds dependencies, runtime cost, and model-domain assumptions. The notebook tries to load a local transformer model if configured.

If no local model is available and downloads are disabled, the notebook uses a clearly labeled fallback. That fallback allows the notebook to run in smoke/offline mode, but it should not be presented as equivalent to a transformer model in the report.

In [ ]:
# TODO (Category A — Modern library): Implement RoBERTa transformer sentiment ranking.
# Estimated time: ~3 h
#
# Implement TWO functions:
#
# ── A) load_transformer_sentiment_pipeline() → (pipeline_or_None, label_string) ─
#   - If USE_TRANSFORMER_SENTIMENT is False: return (None, "disabled").
#   - Try: from transformers import pipeline
#   - If unavailable: return (None, "transformers_unavailable: <error>").
#   - If TRANSFORMER_MODEL_PATH is set: load from local path with local_files_only=True.
#   - Elif ALLOW_TRANSFORMER_DOWNLOAD: load "cardiffnlp/twitter-roberta-base-sentiment-latest"
#     (or the configured TRANSFORMERr MODEL constant).
#   - Else: return (None, "offline_mode_no_cached_model").
#   - Return (clf, descriptive_label_string).
#
# ── B) build_transformer_ranking(evidence, clf) → DataFrame ─────────────────
#   - If clf is None: return an empty DataFrame with the expected columns.
#   - For each evidence row, run clf(text[:MAX_INPUT_CHARS], truncation=True).
#   - Map label to a score: positive → +1, negative → -1, neutral → 0,
#     multiply by the pipeline's score (confidence).
#   - Aggregate per (dish, business_id, business_name): mean transformer score.
#   - Apply shrink_score and mention_bonus; set method = "transformer_sentiment".
#   - Return DataFrame with same schema as ratings_ranking.
#   - Keep batches small (batch_size=16 or less) to avoid OOM on Colab.
#
# After implementing:
#   clf, transformer_method = load_transformer_sentiment_pipeline()
#   transformer_ranking = build_transformer_ranking(evidence, clf)

# 构造离线 fallback 的轻量描述对象。
def _make_transformer_fallback(reason: str):
    # 返回带原因的 fallback 配置字典。
    return {"kind": "fallback", "reason": reason}

# 将 transformer 输出标签映射到 [-1, 1] 情感分数。
def transformer_label_to_score(output) -> float:
    # 如果 pipeline 返回多个标签分数，则选择置信度最高的标签。
    if isinstance(output, list):
        # 空输出按中性分数处理。
        if not output:
            # 返回中性分数。
            return 0.0
        # 选择最高置信度的标签结果。
        output = max(output, key=lambda item: float(item.get("score", 0.0)))
    # 读取标签并统一成小写字符串。
    label = str(output.get("label", "")).lower()
    # 读取模型置信度，缺失时按 0 处理。
    confidence = float(output.get("score", 0.0))
    # 将正向标签映射为正分。
    if "positive" in label or label in {"label_2", "2"}:
        # 返回正向置信度分数。
        return confidence
    # 将负向标签映射为负分。
    if "negative" in label or label in {"label_0", "0"}:
        # 返回负向置信度分数。
        return -confidence
    # 中性或未知标签映射为 0。
    return 0.0

def load_transformer_sentiment_pipeline():
    if not USE_TRANSFORMER_SENTIMENT:
        return None, "disabled"
    # 设置离线 fallback 的方法标签。
    fallback_label = "transformer_fallback_sentiment"
    try:
        from transformers import pipeline as hf_pipeline
    except Exception as exc:
        # transformers 不可用时使用明确标记的离线 fallback。
        return _make_transformer_fallback(f"transformers_unavailable: {exc}"), f"{fallback_label}: transformers_unavailable: {exc}"
    # TODO: handle TRANSFORMER_MODEL_PATH, ALLOW_TRANSFORMER_DOWNLOAD, offline fallback
    # 优先尝试从用户配置的本地模型路径加载。
    if TRANSFORMER_MODEL_PATH:
        # 捕获本地模型路径或权重不完整导致的加载失败。
        try:
            # 从本地路径加载情感分类 pipeline，并禁止隐式下载。
            clf = hf_pipeline("sentiment-analysis", model=TRANSFORMER_MODEL_PATH, tokenizer=TRANSFORMER_MODEL_PATH, local_files_only=True)
            # 返回真实本地 transformer pipeline。
            return clf, f"local_transformer: {TRANSFORMER_MODEL_PATH}"
        # 本地模型加载失败时退回可复现 fallback。
        except Exception as exc:
            # 返回带失败原因的 fallback。
            return _make_transformer_fallback(f"local_model_load_failed: {exc}"), f"{fallback_label}: local_model_load_failed: {exc}"
    # 只有显式允许下载时才访问 Hugging Face 模型。
    if ALLOW_TRANSFORMER_DOWNLOAD:
        # 捕获模型下载或初始化失败。
        try:
            # 加载配置中的 transformer 情感模型。
            clf = hf_pipeline("sentiment-analysis", model=TRANSFORMER_MODEL_ID, tokenizer=TRANSFORMER_MODEL_ID)
            # 返回真实下载 transformer pipeline。
            return clf, f"downloaded_transformer: {TRANSFORMER_MODEL_ID}"
        # 下载或初始化失败时退回可复现 fallback。
        except Exception as exc:
            # 返回带失败原因的 fallback。
            return _make_transformer_fallback(f"download_or_load_failed: {exc}"), f"{fallback_label}: download_or_load_failed: {exc}"
    # 离线模式下使用词典分数和星级先验组成的明确 fallback。
    return _make_transformer_fallback("offline_mode_no_cached_model"), f"{fallback_label}: offline_mode_no_cached_model"

def build_transformer_ranking(evidence: "pd.DataFrame", clf) -> "pd.DataFrame":
    # 使用与其他排名方法一致的输出列。
    EXPECTED_COLS = RANKING_COLUMNS
    if clf is None:
        print("Transformer pipeline unavailable; skipping Method C.")
        return pd.DataFrame(columns=EXPECTED_COLS)
    # TODO: iterate over evidence rows (with truncation); score; aggregate; shrink; rank; return
    # 没有证据时返回统一 schema 的空表。
    if evidence is None or evidence.empty:
        # 构造空排名表以保证后续拼接逻辑稳定。
        return pd.DataFrame(columns=EXPECTED_COLS)
    # 优先使用局部上下文窗口，缺失时退回命中的单句。
    text_col = "context_window" if "context_window" in evidence.columns else "sentence"
    # 限制 transformer 或 fallback 评分的证据行数，保持运行成本可控。
    work = evidence.head(MAX_TRANSFORMER_EVIDENCE_ROWS).copy()
    # 设置单条输入的最大字符数，避免超长文本拖慢推理。
    max_input_chars = int(os.environ.get("TASK4_MAX_INPUT_CHARS", "512"))
    # 判断当前评分器是否为离线 fallback。
    if isinstance(clf, dict) and clf.get("kind") == "fallback":
        # 标记 fallback 方法名称，避免与真实 transformer 混淆。
        method_name = "transformer_fallback_sentiment"
        # 计算 fallback 使用的局部词典情感分数。
        work["fallback_lexicon_score"] = work[text_col].fillna("").map(simple_lexicon_sentiment)
        # 计算 fallback 使用的星级先验分数。
        work["star_unit_score"] = work["stars"].map(star_to_unit_score).fillna(0.0)
        # 用局部文本信号为主、星级先验为辅组合 fallback 分数。
        work["transformer_score"] = (0.70 * work["fallback_lexicon_score"]) + (0.30 * work["star_unit_score"])
    # 真实 transformer pipeline 使用模型输出标签和置信度。
    else:
        # 标记真实 transformer 方法名称。
        method_name = "transformer_sentiment"
        # 准备截断后的模型输入文本列表。
        texts = work[text_col].fillna("").astype(str).str.slice(0, max_input_chars).tolist()
        # 初始化逐条证据的 transformer 分数列表。
        scores = []
        # 使用小批量推理，降低 Colab 或本地 CPU/GPU 的内存压力。
        batch_size = 16
        # 分批运行 transformer pipeline。
        for start in range(0, len(texts), batch_size):
            # 取出当前批次文本。
            batch_texts = texts[start:start + batch_size]
            # 调用 pipeline 并要求 tokenizer 截断超长输入。
            batch_outputs = clf(batch_texts, truncation=True, batch_size=batch_size)
            # 将每条输出映射为数值分数。
            for output in batch_outputs:
                # 保存当前证据的 transformer 情感分数。
                scores.append(transformer_label_to_score(output))
        # 把模型分数写入工作表。
        work["transformer_score"] = scores
    # 将 transformer/fallback 分数写回证据表，供后续 artifact 使用。
    evidence.loc[work.index, "transformer_score"] = work["transformer_score"]
    # 删除没有有效 transformer/fallback 分数的证据行。
    work = work.dropna(subset=["transformer_score"]).copy()
    # 如果没有有效分数，则返回空排名表。
    if work.empty:
        # 构造空排名表以保证后续拼接逻辑稳定。
        return pd.DataFrame(columns=EXPECTED_COLS)
    # 计算 transformer/fallback 全局均值作为收缩先验。
    raw_global_mean = work["transformer_score"].mean()
    # 缺失时使用中性先验零分。
    global_mean = 0.0 if pd.isna(raw_global_mean) else float(raw_global_mean)
    # 汇总每个菜品和餐厅组合的提及数、平均星级和 transformer/fallback 均值。
    grouped = work.groupby(["dish", "business_id", "business_name"], dropna=False).agg(mention_count=("review_id", "count"), avg_stars=("stars", "mean"), mean_score=("transformer_score", "mean")).reset_index()
    # 过滤掉提及数不足的餐厅菜品组合。
    grouped = grouped[grouped["mention_count"] >= MIN_BUSINESS_MENTIONS_FOR_RANKING].copy()
    # 如果过滤后为空，则返回统一 schema 的空表。
    if grouped.empty:
        # 构造空排名表以保证后续拼接逻辑稳定。
        return pd.DataFrame(columns=EXPECTED_COLS)
    # 记录最大提及数用于计算热度奖励。
    max_mentions = int(grouped["mention_count"].max())
    # 计算收缩后的 transformer/fallback 评分并叠加温和的提及数奖励。
    grouped["rank_score"] = grouped.apply(lambda row: shrink_score(row["mean_score"], int(row["mention_count"]), global_mean) + mention_bonus(int(row["mention_count"]), max_mentions), axis=1)
    # 标记当前排名方法名称。
    grouped["method"] = method_name
    # 按菜品、分数降序和 business_id 升序排序以稳定处理并列。
    grouped = grouped.sort_values(["dish", "rank_score", "business_id"], ascending=[True, False, True]).reset_index(drop=True)
    # 在每个菜品内部生成从 1 开始的名次。
    grouped["rank"] = grouped.groupby("dish").cumcount().add(1)
    # 返回统一列顺序的排名表。
    return grouped[EXPECTED_COLS]

# Call:
clf, transformer_method = load_transformer_sentiment_pipeline()
transformer_ranking = build_transformer_ranking(evidence, clf)
print(f"Transformer method: {transformer_method}")
# --- Validation ---
print(f"\u2713 Transformer ranking: {len(transformer_ranking)} rows")


## 9. Compare ranking methods

This section combines the three ranking outputs and compares them. The goal is not to declare one method universally correct. The goal is to understand what each method emphasizes:

- ratings-only uses broad review-level user satisfaction;
- lexicon sentiment uses transparent dish-local sentiment clues;
- transformer sentiment is the modern comparison path and may capture more context, while adding model assumptions and runtime cost.

The ranking tables below filter out restaurant/dish pairs with fewer than `MIN_BUSINESS_MENTIONS_FOR_RANKING` local mentions. This keeps the main recommendations more defensible while retaining sparse evidence in `dish_mention_evidence.csv` and `dish_mention_coverage.csv` for error analysis. Main interpretation should prioritize dishes with `strong` or `moderate` evidence tiers; `sparse` dishes are useful for discussing uncertainty rather than making confident recommendations.

**Expected pattern in the reference real-data run.** Ranking outputs should include all three methods and should not include restaurant/dish pairs below the per-business mention threshold. The transformer column should come from a transformer model in real submissions. If setup smoke/offline validation uses a labeled fallback, that fallback is a reproducibility guardrail, not a true independent transformer result.

Rankings should represent repeated local evidence, not one lucky mention.

In [ ]:
# TODO (Category C — Method design): Merge rankings and compute cross-method agreement.
# Estimated time: ~1.5 h
#
# Step 1 — Merge all three method rankings.
#   Concatenate ratings_ranking, lexicon_ranking, transformer_ranking.
#   Merge with coverage[["dish","evidence_tier","main_analysis_ready"]] on "dish".
#   Save to ARTIFACT_DIR / "dish_restaurant_rankings_long.csv".
#
# Step 2 — Implement Jaccard agreement between top-k ranking lists.
#   topk_set(df, dish, method, k=5) → set[business_id]  top k restaurants
#   jaccard(a, b) → float  |a ∩ b| / |a ∪ b|, return nan if both empty
#
# Step 3 — Build ranking_method_comparison table.
#   For each dish × method-pair combination:
#     Compute jaccard(topk_set(method_A), topk_set(method_B)).
#   Save to ARTIFACT_DIR / "ranking_method_comparison.csv"
#   (columns: dish, method_a, method_b, top5_jaccard).
#   Print: mean jaccard across all dishes.
#
# Step 4 — Build top_recommendations table.
#   For each dish × method, collect the top-3 restaurants (rank, business_name, rank_score).
#   Save to ARTIFACT_DIR / "top_recommendations_by_method.csv"
#   (columns: dish, method, rank, business_name, rank_score, mention_count, avg_stars).
#
# Step 5 — Display a pivot table: dish × method → top-1 restaurant name.
#
# Design note: low jaccard across methods (< 0.3) is interesting, not wrong.
# In your report, explain why methods agree or disagree for specific dishes.

def topk_set(df: "pd.DataFrame", dish: str, method: str, k: int = 5) -> set:
    # TODO: filter by dish and method; sort by rank; return set of top-k business_ids
    # 筛选指定菜品和方法的排名结果。
    sub = df[(df["dish"] == dish) & (df["method"] == method)].sort_values(["rank", "business_id"]).head(k)
    # 返回 top-k 餐厅 business_id 集合。
    return set(sub["business_id"].astype(str))

def jaccard(a: set, b: set) -> float:
    # TODO: |a∩b| / |a∪b|, return nan if a∪b is empty
    # 计算两个集合的并集。
    union = set(a) | set(b)
    # 两个集合都为空时返回缺失值。
    if not union:
        # 空集合没有可解释的一致性比例。
        return np.nan
    # 计算两个集合的交集。
    intersection = set(a) & set(b)
    # 返回交集占并集的比例。
    return len(intersection) / len(union)

# TODO: concatenate and merge rankings
# 收集所有非空的排名结果表。
ranking_frames = [frame for frame in [ratings_ranking, lexicon_ranking, transformer_ranking] if isinstance(frame, pd.DataFrame) and not frame.empty]
# 合并长表；没有任何排名时保留统一 schema。
rankings = pd.concat(ranking_frames, ignore_index=True) if ranking_frames else pd.DataFrame(columns=RANKING_COLUMNS)
# 如果覆盖诊断可用，则把证据层级并入排名表。
if not rankings.empty and "coverage" in globals() and isinstance(coverage, pd.DataFrame) and not coverage.empty:
    # 选择与排名解释相关的覆盖诊断列。
    coverage_cols = ["dish", "evidence_tier", "main_analysis_ready"]
    # 将覆盖诊断列合并到每条排名记录。
    rankings = rankings.merge(coverage[coverage_cols], on="dish", how="left")
# 记录当前排名表中实际可比较的方法列表。
methods = sorted(rankings["method"].dropna().unique().tolist()) if not rankings.empty else []

# TODO: save rankings
# rankings.to_csv(ARTIFACT_DIR / "dish_restaurant_rankings_long.csv", index=False)
# 保存长格式餐厅排名表。
rankings.to_csv(ARTIFACT_DIR / "dish_restaurant_rankings_long.csv", index=False)

# TODO: build ranking_method_comparison table and save
# 初始化方法一致性记录列表。
comparison_rows = []
# 遍历每个菜品计算方法两两 top-5 一致性。
for dish in sorted(rankings["dish"].dropna().unique()) if not rankings.empty else []:
    # 遍历第一个方法索引。
    for i, method_a in enumerate(methods):
        # 遍历第二个方法，避免重复组合。
        for method_b in methods[i + 1:]:
            # 取出方法 A 的 top-5 餐厅集合。
            set_a = topk_set(rankings, dish, method_a, k=5)
            # 取出方法 B 的 top-5 餐厅集合。
            set_b = topk_set(rankings, dish, method_b, k=5)
            # 保存当前菜品和方法对的一致性分数。
            comparison_rows.append({"dish": dish, "method_a": method_a, "method_b": method_b, "top5_jaccard": jaccard(set_a, set_b)})
# 构造方法一致性 DataFrame。
method_comparison = pd.DataFrame(comparison_rows, columns=["dish", "method_a", "method_b", "top5_jaccard"])
# 保存方法一致性 artifact。
method_comparison.to_csv(ARTIFACT_DIR / "ranking_method_comparison.csv", index=False)
# 计算所有可用方法对的平均 Jaccard。
mean_jaccard = method_comparison["top5_jaccard"].mean() if not method_comparison.empty else np.nan
# 打印方法一致性摘要。
print(f"Mean top-5 Jaccard agreement: {mean_jaccard:.3f}" if not pd.isna(mean_jaccard) else "Mean top-5 Jaccard agreement: n/a")

# TODO: build top_recommendations_by_method table and save
# 定义 report-ready top recommendations 的列顺序。
top_rec_cols = ["dish", "method", "rank", "business_name", "rank_score", "mention_count", "avg_stars"]
# 初始化空的 top recommendations 表。
top_recommendations = pd.DataFrame(columns=top_rec_cols)
# 当排名表非空时抽取每个菜品和方法的前三名。
if not rankings.empty:
    # 按菜品、方法和名次稳定排序。
    top_recommendations = rankings.sort_values(["dish", "method", "rank", "business_id"]).groupby(["dish", "method"], group_keys=False).head(3).copy()
    # 只保留报告要求的列。
    top_recommendations = top_recommendations[top_rec_cols]
# 保存 top recommendations artifact。
top_recommendations.to_csv(ARTIFACT_DIR / "top_recommendations_by_method.csv", index=False)
# 当排名表非空时展示每个菜品和方法的第一名餐厅。
if not rankings.empty:
    # 构造菜品 × 方法的 top-1 餐厅透视表。
    top1_pivot = rankings[rankings["rank"] == 1].pivot_table(index="dish", columns="method", values="business_name", aggfunc="first")
    # 展示 top-1 对照表。
    display(top1_pivot)
# 展示方法一致性表预览。
display(method_comparison.head(20))
# 展示 report-ready top recommendations 预览。
display(top_recommendations.head(20))

# --- Validation ---
assert isinstance(rankings, pd.DataFrame) and "method" in rankings.columns
assert (ARTIFACT_DIR / "dish_restaurant_rankings_long.csv").exists()
assert (ARTIFACT_DIR / "ranking_method_comparison.csv").exists()
assert (ARTIFACT_DIR / "top_recommendations_by_method.csv").exists()
print(f"\u2713 Rankings: {len(rankings)} rows, {rankings['method'].nunique()} methods, {rankings['dish'].nunique()} dishes")

In [ ]:
# Save core Task 4 handoff artifacts.
if "mention_evidence" in globals() and not mention_evidence.empty:
    mention_evidence.to_csv(ARTIFACT_DIR / "dish_mention_evidence.csv", index=False)
    print(f"Saved: dish_mention_evidence.csv ({len(mention_evidence)} rows)")

if "coverage" in globals() and not coverage.empty:
    coverage.to_csv(ARTIFACT_DIR / "dish_mention_coverage.csv", index=False)
    print(f"Saved: dish_mention_coverage.csv ({len(coverage)} rows)")

if "rankings" in globals() and not rankings.empty:
    rankings.to_csv(ARTIFACT_DIR / "dish_restaurant_rankings_long.csv", index=False)
    print(f"Saved: dish_restaurant_rankings_long.csv ({len(rankings)} rows)")

## Evidence-Grounded Recommendation Explanations

Rankings should be explainable. This section selects the top-ranked restaurants for a few
dishes and retrieves the review snippets that justify each ranking. The result is a short
**grounded explanation** — a claim backed by specific retrieved evidence.

This connects to the idea behind Retrieval-Augmented Generation (evidence-grounded explanation): conclusions should be
grounded in retrieved evidence, not asserted without support. Here we do it deterministically,
using the dish-local evidence already extracted in the ranking pipeline.

In [ ]:
def build_grounded_explanations(
    evidence_df: pd.DataFrame,
    rankings_df: pd.DataFrame,
    top_k_dishes: int = 3,
    top_k_restaurants: int = 2,
    top_k_snippets: int = 5,
) -> pd.DataFrame:
    """
    For top-ranked (dish, restaurant, method) pairs, retrieve supporting
    evidence snippets and produce a deterministic grounded explanation.

    Returns DataFrame with columns:
    dish, business_id, business_name, ranking_method, rank,
    evidence_snippets, grounded_explanation, evidence_support_rating.
    """
    if evidence_df.empty or rankings_df.empty:
        return pd.DataFrame()

    rows = []

    # Top dishes by mention count.
    top_dishes = (
        evidence_df.groupby("dish")
        .size()
        .nlargest(top_k_dishes)
        .index.tolist()
    )

    for dish in top_dishes:
        dish_rankings = rankings_df[rankings_df["dish"] == dish].copy()
        if dish_rankings.empty:
            continue

        # Keep grounded explanations balanced across ranking methods.
        # top_k_restaurants is interpreted per method, so lexicon results are not
        # dropped when ratings and transformer share the same rank value.
        if "method" in dish_rankings.columns:
            sort_cols = [c for c in ["method", "rank", "business_id"] if c in dish_rankings.columns]
            selected_rankings = (
                dish_rankings.sort_values(sort_cols)
                .groupby("method", group_keys=False)
                .head(top_k_restaurants)
            )
        else:
            selected_rankings = (
                dish_rankings.sort_values("rank")
                .head(top_k_restaurants)
            )

        for _, rank_row in selected_rankings.iterrows():
            bid      = rank_row.get("business_id")
            biz_name = rank_row.get("business_name", str(bid))
            method   = rank_row.get("method", "unknown")
            rank_val = rank_row.get("rank", float("nan"))

            ev_sub = evidence_df[
                (evidence_df["dish"] == dish) &
                (evidence_df["business_id"] == bid)
            ].head(top_k_snippets)

            if ev_sub.empty:
                continue

            context_col = next(
                (c for c in ["context_window", "sentence"] if c in ev_sub.columns), None
            )
            snippets = (
                ev_sub[context_col].fillna("").tolist()
                if context_col else []
            )
            snippets_text = " | ".join(
                s[:100] + "..." if len(s) > 100 else s for s in snippets
            )

            # Simple evidence support rating from lexicon sentiment.
            avg_sent = 0.0
            if "lexicon_score" in ev_sub.columns:
                avg_sent = float(ev_sub["lexicon_score"].mean())
            elif "stars" in ev_sub.columns:
                avg_sent = (float(ev_sub["stars"].mean()) - 3.0) / 2.0  # map 1-5 → -1..1

            if avg_sent > 0.3:
                support = "strong support"
            elif avg_sent > 0.0:
                support = "partial support"
            elif avg_sent >= -0.15:
                support = "weak support"
            else:
                support = "contradictory evidence"

            # Derive top positive theme words from snippets.
            _theme_counts: dict = {}
            for _snip in snippets:
                for _w in re.findall(r'\b\w+\b', _snip.lower()):
                    if _w in POSITIVE_WORDS:
                        _theme_counts[_w] = _theme_counts.get(_w, 0) + 1
            _top_themes = sorted(_theme_counts, key=_theme_counts.get, reverse=True)[:3]
            _n_pos = sum(1 for s in snippets if any(w in s.lower() for w in POSITIVE_WORDS))

            _theme_clause = (
                f"Snippet language centers on: {', '.join(_top_themes)}."
                if _top_themes else "Snippets contain limited positive signal."
            )
            explanation = (
                f"{biz_name} is ranked #{int(rank_val) if not pd.isna(rank_val) else '?'} "
                f"for {dish} by {method}. "
                f"{_n_pos} of {len(ev_sub)} snippet(s) contain positive language. "
                f"{_theme_clause}"
            )

            rows.append({
                "dish":                  dish,
                "business_id":           bid,
                "business_name":         biz_name,
                "ranking_method":        method,
                "rank":                  rank_val,
                "evidence_snippets":     snippets_text,
                "grounded_explanation":  explanation,
                "evidence_support_rating": support,
            })

    return pd.DataFrame(rows)


if "mention_evidence" in globals() and "rankings" in globals():
    grounded_explanations = build_grounded_explanations(mention_evidence, rankings)
    if not grounded_explanations.empty:
        grounded_explanations.to_csv(
            ARTIFACT_DIR / "grounded_recommendation_explanations.csv", index=False
        )
        print("Saved grounded_recommendation_explanations.csv")
        display(grounded_explanations)
    else:
        print("No grounded explanations generated (evidence or rankings unavailable).")
        grounded_explanations = pd.DataFrame()
else:
    grounded_explanations = pd.DataFrame()
    print("Skipping grounded explanations (mention_evidence or rankings not available).")

In [ ]:
# Readable per-dish evidence display for student reports.
# The DataFrame above is compact; this block shows each snippet on its own line.
if "grounded_explanations" in dir() and not grounded_explanations.empty:
    for _, row in grounded_explanations.iterrows():
        print(f"\n{'='*60}")
        print(f"Dish: {row['dish']}  |  Restaurant: {row['business_name']}")
        print(f"Method: {row['ranking_method']}  |  Rank: #{int(row['rank']) if not pd.isna(row['rank']) else '?'}")
        print(f"Support: {row['evidence_support_rating']}")
        print(f"Explanation: {row['grounded_explanation']}")
        print("\nEvidence snippets:")
        for j, snip in enumerate(str(row['evidence_snippets']).split(' | '), 1):
            if snip.strip():
                print(f"  [{j}] {snip.strip()}")
else:
    print("No grounded explanations available.")


### Interpreting grounded explanations

For each (dish, restaurant) pair in the table above:
- Do the retrieved snippets actually support the ranking?
- Is the evidence **strong** (consistently positive dish mentions) or **partial/weak**?
- Are there cases where the ranking method and the snippet evidence disagree?
  That disagreement is itself an important finding.

In [ ]:
def plot_top_restaurants_for_dish(rankings: pd.DataFrame, dish: str, method: str, top_k: int = 8):
    sub = rankings[(rankings["dish"] == dish) & (rankings["method"] == method)].sort_values("rank").head(top_k)
    if sub.empty:
        return None
    fig, ax = plt.subplots(figsize=(8, max(3.5, 0.45 * len(sub))))
    y = np.arange(len(sub))
    ax.barh(y, sub["rank_score"])
    ax.set_yticks(y)
    ax.set_yticklabels(sub["business_name"])
    ax.invert_yaxis()
    ax.set_xlabel("Rank score")
    ax.set_title(f"Top restaurants for '{dish}' — {method}")
    for i, (_, row) in enumerate(sub.iterrows()):
        ax.text(row["rank_score"], i, f"  n={int(row['mention_count'])}", va="center", fontsize=8)
    fig.tight_layout()
    out = ARTIFACT_DIR / f"top_restaurants_{re.sub(r'[^a-z0-9]+', '_', dish.lower()).strip('_')}_{method}.png"
    fig.savefig(out, dpi=160, bbox_inches="tight")
    plt.show()
    return out

plot_paths = []
for dish in SELECTED_DISHES_FOR_PLOTS:
    if dish in set(rankings["dish"]):
        for method in methods:
            p = plot_top_restaurants_for_dish(rankings, dish, method, TOP_K_PER_DISH)
            if p:
                plot_paths.append(str(p))

pd.DataFrame({"plot_path": plot_paths}).to_csv(ARTIFACT_DIR / "saved_plot_paths.csv", index=False)
print("Saved plots:", len(plot_paths))

## 10. Qualitative error analysis

A ranking table alone is not enough. You should inspect the evidence behind at least a few recommendations.

Useful questions:

- Is the sentence actually about the target dish?
- Is sentiment about the dish, the restaurant, or something else?
- Did the method handle negation or contrast?
- Does a high star rating conflict with dish-local text?
- Are there too few mentions to trust the rank?

In [ ]:
# TODO (Category D — Medium pipeline): Sample and display dish-level error analysis.
# Estimated time: ~1 h
#
# Implement sample_evidence_for_top_rankings(evidence, rankings, k_dishes, k_restaurants, k_snippets).
# - For the top-k_dishes dishes by evidence count:
#   - For each dish, pick top-k_restaurants by each method.
#   - For each (dish, restaurant, method) triple, collect up to k_snippets evidence snippets.
#   - Return a DataFrame [dish, business_name, method, rank, snippet].
#
# After calling it, display:
#   1. The evidence sample table.
#   2. For 2–3 dishes, show where methods disagree (different top-ranked restaurants).
#      A disagreement exists when topk_set(method_A) ∩ topk_set(method_B) is small.
#
# This is qualitative error analysis — no formula needed, just manual inspection.

def sample_evidence_for_top_rankings(evidence, rankings, k_dishes=3, k_restaurants=2, k_snippets=5):
    # TODO: select top-k dishes; for each, retrieve supporting snippets from evidence
    # 没有证据或排名时返回统一 schema 的空表。
    if evidence is None or rankings is None or evidence.empty or rankings.empty:
        # 构造空样本表，避免后续 display 报错。
        return pd.DataFrame(columns=["dish", "business_name", "method", "rank", "snippet"])
    # 优先展示局部上下文窗口，缺失时退回命中单句。
    context_col = "context_window" if "context_window" in evidence.columns else "sentence"
    # 选择证据最多的若干菜品。
    top_dishes = evidence.groupby("dish").size().sort_values(ascending=False).head(k_dishes).index.tolist()
    # 初始化证据样本记录列表。
    rows = []
    # 遍历高覆盖菜品。
    for dish in top_dishes:
        # 取出当前菜品的所有方法排名。
        dish_rankings = rankings[rankings["dish"] == dish].sort_values(["method", "rank", "business_id"])
        # 对每个方法保留前 k_restaurants 个餐厅。
        top_rankings = dish_rankings.groupby("method", group_keys=False).head(k_restaurants)
        # 遍历待检查的菜品、餐厅、方法组合。
        for _, rank_row in top_rankings.iterrows():
            # 筛选当前餐厅对当前菜品的支持证据。
            ev_sub = evidence[(evidence["dish"] == dish) & (evidence["business_id"] == rank_row["business_id"])].head(k_snippets)
            # 遍历最多 k_snippets 条证据。
            for _, ev_row in ev_sub.iterrows():
                # 保存一条可读的证据样本记录。
                rows.append({"dish": dish, "business_name": rank_row["business_name"], "method": rank_row["method"], "rank": rank_row["rank"], "snippet": ev_row.get(context_col, "")})
    # 返回证据样本表。
    return pd.DataFrame(rows)

evidence_sample = sample_evidence_for_top_rankings(evidence, rankings)
if isinstance(evidence_sample, pd.DataFrame) and not evidence_sample.empty:
    display(evidence_sample)

# TODO: for 2–3 dishes, print a short analysis of where methods agree/disagree
# Example:
#   For dish X: ratings ranks Biz A #1, but lexicon ranks Biz B #1.
#   Reading the evidence snippets: Biz A has high stars but bland mentions.
# 如果有排名结果，则自动生成简短的 top-1 同异摘要。
if isinstance(rankings, pd.DataFrame) and not rankings.empty:
    # 选择证据最多的 3 个菜品用于文字分析。
    analysis_dishes = evidence.groupby("dish").size().sort_values(ascending=False).head(3).index.tolist()
    # 遍历每个待分析菜品。
    for dish in analysis_dishes:
        # 取出当前菜品各方法的第一名。
        top1 = rankings[(rankings["dish"] == dish) & (rankings["rank"] == 1)].sort_values("method")
        # 没有第一名时跳过当前菜品。
        if top1.empty:
            # 继续处理下一个菜品。
            continue
        # 生成方法到第一名餐厅的简短描述。
        summary = "; ".join(f"{row.method}: {row.business_name}" for row in top1.itertuples(index=False))
        # 判断各方法第一名是否一致。
        agreement_note = "methods agree on the top restaurant" if top1["business_id"].nunique() == 1 else "methods disagree on the top restaurant"
        # 打印当前菜品的方法同异摘要。
        print(f"For {dish}: {summary}. Overall, {agreement_note}.")


## Expected kinds of evidence to inspect

Strong Task 4 reports use concrete examples: a dish with enough mentions, a restaurant that ranks highly across methods, a restaurant whose rank changes across methods, and at least one local sentence that explains the disagreement. Agreement across methods can increase confidence, but sparse or noisy evidence should still be labeled cautiously.

## Visual Evidence and Interpretation

Visualizations in this notebook are evidence, not decoration. Each figure should support interpretation, method comparison, diagnostic analysis, or communication of a discovered Yelp restaurant-intelligence pattern.

In a student report, explain what each selected figure reveals and one limitation of the visualization, especially when smoke-test data are small or sparse.

### Visual setup

The visual layer uses compact, notebook-native figures. Each chart saves both a PNG and, when useful, the plotting dataframe so students can trace a figure back to the rows that produced it.

In [ ]:
# Visual evidence layer: dish coverage, top rankings, method shifts, and method agreement.
import textwrap
VIS_TOP_K_RESTAURANTS = 8
VIS_MAX_LABELS = 10

def _short_label(value, width=26):
    return "\n".join(textwrap.wrap(str(value), width=width)) or str(value)

### Coverage versus sentiment

This bubble chart asks whether a dish has enough evidence to support ranking, not merely whether it has a high average score. A dish can be valid for Task 3 but too sparse for confident Task 4 ranking.

In [ ]:
if "coverage" in globals() and isinstance(coverage, pd.DataFrame) and not coverage.empty:
    bubble_df = coverage.copy()
    if "mention_evidence" in globals() and isinstance(mention_evidence, pd.DataFrame) and not mention_evidence.empty:
        sentiment_cols = [c for c in ["lexicon_score", "transformer_score", "stars"] if c in mention_evidence.columns]
        agg_spec = {c: (c, "mean") for c in sentiment_cols}
        agg_spec["num_businesses_with_mentions_check"] = ("business_id", "nunique")
        sent_by_dish = mention_evidence.groupby("dish").agg(**agg_spec).reset_index()
        bubble_df = bubble_df.merge(sent_by_dish, on="dish", how="left", suffixes=("", "_avg"))
    y_col = "lexicon_score" if "lexicon_score" in bubble_df.columns else ("stars" if "stars" in bubble_df.columns else None)
    x_col = "mention_count" if "mention_count" in bubble_df.columns else None
    size_col = "num_businesses_with_mentions" if "num_businesses_with_mentions" in bubble_df.columns else "num_businesses_with_mentions_check"
    bubble_df.to_csv(ARTIFACT_DIR / "task4_dish_coverage_sentiment_bubble_values.csv", index=False)
    if x_col and y_col and size_col in bubble_df.columns:
        plot_df = bubble_df.dropna(subset=[x_col, y_col]).copy()
        if len(plot_df) >= 1:
            sizes = 80 + 45 * plot_df[size_col].fillna(1).clip(lower=1)
            fig, ax = plt.subplots(figsize=(8, 5))
            tiers = plot_df["evidence_tier"].fillna("unknown") if "evidence_tier" in plot_df else pd.Series("unknown", index=plot_df.index)
            for tier, group in plot_df.groupby(tiers):
                ax.scatter(group[x_col], group[y_col], s=sizes.loc[group.index], alpha=0.65, label=str(tier))
            label_df = plot_df.sort_values([x_col, size_col], ascending=False).head(VIS_MAX_LABELS)
            for _, row in label_df.iterrows():
                ax.annotate(_short_label(row["dish"], 16), (row[x_col], row[y_col]), fontsize=8, xytext=(4, 4), textcoords="offset points")
            ax.set_title("Dish coverage versus average local sentiment")
            ax.set_xlabel("dish mention count")
            ax.set_ylabel("average local sentiment" if y_col != "stars" else "average stars")
            ax.legend(title="evidence tier", fontsize=8)
            fig.tight_layout()
            fig.savefig(ARTIFACT_DIR / "task4_dish_coverage_sentiment_bubble.png", dpi=160, bbox_inches="tight")
            plt.show()
        else:
            print("Skipping dish coverage bubble chart: no non-missing sentiment values.")
    else:
        print("Skipping dish coverage bubble chart: required coverage columns are missing.")
else:
    print("Skipping dish coverage bubble chart: coverage is unavailable or empty.")

### Representative top-restaurant ranking

Top-k charts are useful only when paired with evidence thresholds. Rankings should represent repeated local evidence, not one lucky mention.

In [ ]:
# TODO (Category D — Medium pipeline): Plot top-restaurant rankings for selected dishes.
# Estimated time: ~0.5 h
#
# Use plot_top_restaurants_for_dish(rankings, dish, method, top_k=8) for:
#   - The dish with the most evidence (from coverage.sort_values("evidence_count").iloc[-1])
#   - One additional dish of your choice
#   - Both ratings_baseline and lexicon_sentiment methods
#
# Save each plot to ARTIFACT_DIR / f"top_restaurants_{dish}_{method}.png".

if "rankings" in globals() and isinstance(rankings, pd.DataFrame) and not rankings.empty:
    # TODO: pick the dish with the most evidence; plot for ratings_baseline and lexicon_sentiment
    # 从覆盖表中选择证据数量最多的菜品。
    top_dish = coverage.sort_values("evidence_count").iloc[-1]["dish"] if "coverage" in globals() and isinstance(coverage, pd.DataFrame) and not coverage.empty else rankings["dish"].iloc[0]

    # TODO: repeat for a second dish of your choice
    # 按证据数量降序生成第二个菜品候选列表。
    second_candidates = [dish for dish in coverage.sort_values("evidence_count", ascending=False)["dish"].tolist() if dish != top_dish] if "coverage" in globals() and isinstance(coverage, pd.DataFrame) and not coverage.empty else []
    # 选择第二个菜品；没有候选时跳过。
    second_dish = second_candidates[0] if second_candidates else None
    # 汇总需要绘图的菜品列表。
    plot_dishes = [dish for dish in [top_dish, second_dish] if isinstance(dish, str) and dish]
    # 遍历代表性菜品。
    for plot_dish in plot_dishes:
        # 分别绘制评分基线和词典情感方法。
        for method in ["ratings_baseline", "lexicon_sentiment"]:
            # 调用已有绘图函数；函数会保存 PNG 并返回路径。
            plot_path = plot_top_restaurants_for_dish(rankings, plot_dish, method, top_k=8)
            # 有图像输出时打印保存路径。
            if plot_path is not None:
                # 打印保存的图像路径。
                print(f"Saved: {plot_path}")


## 11. Required comparison and interpretation

The final report should answer the following questions with concrete examples:

- **Why dish-specific ranking differs** from overall restaurant rating.
- **Three-method comparison**: how do ratings-only, lexicon sentiment, and
  transformer/fallback sentiment differ for a specific dish?
- **Agreement case**: one dish where all methods agree on the top restaurant.
- **Disagreement case**: one dish where methods disagree; what might explain it?
- **Evidence grounding**: for your top recommendation, does the retrieved evidence
  actually support the ranking? Rate the support and explain.

**Insights! 💡**

Task 4's three-method comparison maps onto the historical progression of how NLP researchers have measured *opinion* in text. Here is a brief history of sentiment analysis eras:

**Era 1 — The Dictionary Era (1990s – early 2000s):** Rule-Based Lexicon Sentiment
- **The Tech:** Manually curated opinion lexicons (SentiWordNet, LIWC, opinion dictionaries) with fixed positive/negative word scores.
- **The Vibe:** "Amazing" scores +2; "terrible" scores −2; average the scores in the review. No learning, no context. This is the method ancestor of VADER — the lexicon method you used in this task.

**Era 2 — The Machine Learning Era (2003 – 2013):** Supervised Bag-of-Words
- **The Tech:** Naive Bayes, SVM, and logistic regression trained on labeled movie/product reviews. Pang & Lee (2002) showed that supervised classification could outperform simple lexicons on movie reviews.
- **The Gap:** These models still relied on bag-of-words features. Word order and dish-local context were ignored — a limitation directly visible when you compare whole-review star ratings against sentence-window sentiment.

**Era 3 — The Embedding Era (2014 – 2018):** Word2Vec + Recurrent Networks
- **The Tech:** GloVe/Word2Vec embeddings, BiLSTM sentiment models, and aspect-based sentiment analysis (ABSA).
- **The Insight:** "This restaurant has great service but the dim sum was soggy" requires understanding that *dim sum was soggy* is negative for the dish dimension, even if the overall review is neutral. ABSA was the field's first systematic attempt to solve exactly the dish-local scoring problem you are working on.

**Era 4 — The Transformer Era (2018 – present):** Fine-Tuned BERT/RoBERTa
- **The Tech:** Pretrained transformer models (BERT, RoBERTa, DistilBERT) fine-tuned on sentiment datasets. The `cardiffnlp/twitter-roberta-base-sentiment` model you used in this task is a RoBERTa variant fine-tuned on social-media posts.
- **The Gap:** Training domain matters. A model fine-tuned on tweets may not fully capture the style of Yelp restaurant reviews — which is why comparing it against the lexicon baseline in your rankings is valuable.

**The Updated Timeline for Opinion Mining:**
- Lexicon/Dictionary Era → Supervised ML Era → Embedding/ABSA Era → Transformer/LLM Era
- Each transition improved context sensitivity but added computation, opacity, and domain-shift risk.
- Your dish-local evidence design (sentence windows around mentions) is a practical engineering choice that every era of sentiment analysis requires: the model is only as good as the evidence it receives.


## 12. Required report deliverables

Your PDF report should be approximately **2–3 pages** and include:

- **Dish selection**: which dishes have enough evidence for reliable ranking?
- **Method comparison**: ratings-only vs. lexicon vs. transformer — where do they agree/disagree?
- **Evidence grounding**: one example where retrieved snippets support the top recommendation.
- **Limitations**: a dish or case where the ranking is likely unreliable and why.

## Final artifact validation

Run this cell before submission. It checks that required artifacts exist, are non-empty, and satisfy the most important schema expectations. Optional artifacts are reported but do not block submission.

In [ ]:
expected_artifacts = [
    "loaded_task3_dish_list.csv",
    "dish_alias_table.csv",
    "dish_mention_evidence.csv",
    "dish_mention_coverage.csv",
    "dish_restaurant_rankings_long.csv",
    "ranking_method_comparison.csv",
    "top_recommendations_by_method.csv",
    "grounded_recommendation_explanations.csv",
]
missing = [f for f in expected_artifacts if not (ARTIFACT_DIR / f).exists()]
if missing:
    print("⚠  Missing artifacts:", missing)
else:
    print("✓ All required artifacts present.")
for f in expected_artifacts:
    p = ARTIFACT_DIR / f
    if p.exists():
        try:
            nrows = len(pd.read_csv(p))
            print(f"  OK: {f}: {nrows} rows")
        except Exception:
            print(f"  OK: {f} (unreadable)")
    else:
        print(f"  MISSING: {f}")

## Common pitfalls to avoid

- Ranking restaurants for a dish with too few mentions.
- Treating whole-review stars as direct dish quality without caveats.
- Running sentiment on full reviews instead of dish-local evidence.
- Describing transformer fallback scores as equivalent to a transformer model.
- Ignoring phrase matching errors in dish-local evidence.
- Reporting rankings without evidence snippets.
- Forgetting to save `dish_search_evidence_index.csv` for Task 5.